# Block 1

In [ ]:
# ==========================
# Block 1: Environment Setup & Google Drive Mount
# Version: v3.0
# Purpose: Install pinned dependencies, mount Drive, define all path constants,
#          HARD-ASSERT every path lives under the expected Finetune root,
#          scan and log any stray checkpoint dirs sitting at Drive root,
#          fix RNG seeds, configure logging, and authenticate HuggingFace.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (Dependency Pinning ✅): Unchanged from v2.0. Checks exact versions
#   of torch (2.5.1+cu121), trl (0.13.0), and peft (0.15.0). If all match, skips
#   the ~5 min install. If any mismatch, installs all 11 packages in mandatory
#   order and sends SIGKILL to force kernel restart so new .so files are loaded.
#
# - Feature 2 (Drive Mount ✅): Mounts at /content/drive. Unchanged from v2.0.
#   Mount must happen BEFORE any path constant is defined — Python evaluates
#   os.path.join at assignment time, so the strings are just strings until Drive
#   is live. Drive availability is validated immediately after mount.
#
# - Feature 3 (Path Constants ✅): Unchanged from v2.0 except one addition:
#   _FINETUNE_ROOT is defined as the single authoritative root all other paths
#   must descend from. Every path is built via os.path.join from BASE_DIR.
#   Paths are absolute strings — no relative components, no trailing slashes.
#
# - Feature 4 (Path Integrity Guard ✅ — NEW in v3.0): _verify_paths() iterates
#   every critical constant (CHECKPOINT_DIR, BEST_MODEL_DIR, ADAPTER_DIR,
#   LOG_DIR, RESULTS_DIR, RAW_DATA_PATH) and asserts each starts with
#   _FINETUNE_ROOT. If any path is outside the expected root — which would
#   happen if BASE_DIR was accidentally set to "/content/drive/MyDrive" instead
#   of the full Twitter/Finetune path — the function raises a RuntimeError with
#   the exact offending path before any directory is created or any model is
#   touched. This prevents the root-spam issue at source.
#
# - Feature 5 (Root Debris Scanner ✅ — NEW in v3.0): _scan_root_debris()
#   lists /content/drive/MyDrive directly and logs every directory whose name
#   starts with "checkpoint-" or matches model folder names (qwen3.5_4b,
#   qwen3.5_9b). These are artefacts from previous sessions where CHECKPOINT_DIR
#   resolved to the Drive root. The scanner logs them with size estimates so
#   the user knows what to delete manually. It does NOT delete automatically
#   because those dirs might be from other projects.
#
# - Feature 6 (Reproducibility ✅): Seeds Python random, NumPy, PyTorch CPU and
#   CUDA all to SEED=42. Unchanged from v2.0.
#
# - Feature 7 (Logging ✅): Dual-handler logger "dei_finetune" writing to both
#   training_4b.log on Drive (append mode) and stdout. Unchanged from v2.0.
#
# - Feature 8 (HuggingFace Auth ✅): Reads HF_TOKEN from Colab Secrets via
#   userdata.get(). Raises clear RuntimeError if missing. Unchanged from v2.0.
#
# How it works:
# - Step 1: Version-check three anchor packages.
# - Step 2: If mismatch, install all packages and SIGKILL for restart.
# - Step 3: Mount Drive, import libraries.
# - Step 4: Define _FINETUNE_ROOT and all path constants.
# - Step 5: Call _verify_paths() — raises on any off-root path.
# - Step 6: Call _scan_root_debris() — logs stray dirs at Drive root.
# - Step 7: Create directories, configure logger, set seeds, verify CUDA, auth HF.
#
# Solutions attempted that did not work:
# - v2.0: No path verification — if BASE_DIR was wrong (e.g. truncated to just
#   /content/drive/MyDrive in a prior session), all saves landed at Drive root
#   and no error was raised. Fixed in v3.0 by _verify_paths().
# - v2.0: No debris scanner — root-level checkpoint-N and qwen3.5_4b dirs
#   accumulated silently across sessions. Fixed in v3.0 by _scan_root_debris().
#
# Solutions implemented:
# - _FINETUNE_ROOT constant as single source of truth for path validation.
# - _verify_paths() hard-assert before any os.makedirs call.
# - _scan_root_debris() to surface existing root-level debris for manual cleanup.
# ==========================
# ==========================
# Block 1: Environment Setup & Google Drive Mount
# Version: v3.2 (No-Stop + Upfront Auth)
# Purpose: Upfront Drive & HF authentication, install pinned dependencies,
#          HARD-ASSERT every path lives under the expected Finetune root,
#          scan and log any stray checkpoint dirs sitting at Drive root,
#          fix RNG seeds, configure logging, and authenticate HuggingFace.
# ==========================

print("====++++++====")
print("Block 1 Output")
print("====++++++====")
print()

import os, sys, subprocess, getpass
from google.colab import drive, userdata

# ── 1. UPFRONT AUTHENTICATION & PERMISSIONS ────────────────────────────────
print("⏳ Step 1: Checking permissions (Drive & HuggingFace)...")

# A. Mount Google Drive immediately so the prompt appears right away
drive.mount("/content/drive")

_drive_root = "/content/drive/MyDrive"
if not os.path.isdir(_drive_root):
    raise RuntimeError(
        "Drive mount failed — /content/drive/MyDrive not found. "
        "Re-run this cell after confirming Drive access."
    )

# B. Retrieve HuggingFace Token (from Secrets or prompt user immediately)
try:
    _hf_token = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN retrieved automatically from Colab Secrets.")
except Exception:
    print("⚠️ HF_TOKEN not found in Colab Secrets.")
    _hf_token = getpass.getpass("🔑 Please paste your HuggingFace token (starts with hf_): ")
    if not _hf_token or not _hf_token.startswith("hf_"):
        raise ValueError("Invalid token provided. It should start with 'hf_'.")

os.environ["HF_TOKEN"] = _hf_token

# ── 2. DEPENDENCY INSTALLATION ─────────────────────────────────────────────
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
         "--no-warn-script-location", *args]
    )

def _check_version(pkg_name: str, required: str) -> bool:
    try:
        import importlib.metadata
        return importlib.metadata.version(pkg_name) == required
    except Exception:
        return False

_installs_needed = (
    not _check_version("torch",  "2.5.1+cu121") or
    not _check_version("trl",    "0.13.0")       or
    not _check_version("peft",   "0.15.0")
)

if _installs_needed:
    print("\n⏳ Installing / upgrading packages — this takes ~5 min...")
    print("   (You have already granted permissions, so you can safely step away!)")
    _pip("torchvision==0.20.1+cu121", "--index-url",
         "https://download.pytorch.org/whl/cu121")
    _pip("torch==2.5.1+cu121", "--index-url",
         "https://download.pytorch.org/whl/cu121")
    _pip("transformers", "--upgrade")
    _pip("peft==0.15.0")
    _pip("trl==0.13.0")
    _pip("bitsandbytes>=0.46.0")
    _pip("accelerate", "--upgrade")
    _pip("datasets==3.2.0")
    _pip("scikit-learn==1.5.2")
    _pip("datasketch==1.6.5")
    _pip("polars==1.17.0")

    print("✅ Packages updated. Refreshing sys.path to continue without stopping...")
    # Refresh system paths so Python recognizes the newly installed packages
    import site
    site.main()
else:
    print("\n✅ All packages already at correct versions — skipping install.")

# Now that dependencies are guaranteed, import the rest
import logging, random, numpy as np, torch
import huggingface_hub

# ── 3. PATH CONSTANTS ──────────────────────────────────────────────────────
# _FINETUNE_ROOT is the single gatekeeper. Every other path MUST start with it.
_FINETUNE_ROOT  = "/content/drive/MyDrive/Twitter/Finetune"
BASE_DIR        = _FINETUNE_ROOT          # alias kept for downstream compatibility

DATA_DIR        = os.path.join(BASE_DIR, "data")
RAW_DATA_PATH   = os.path.join(DATA_DIR, "processed", "merged_4k_v2_train.csv")
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "checkpoints", "qwen3.5_4b")
BEST_MODEL_DIR  = os.path.join(CHECKPOINT_DIR, "best_model")
ADAPTER_DIR     = os.path.join(BASE_DIR, "adapters",    "qwen3.5_4b")
LOG_DIR         = os.path.join(BASE_DIR, "logs")
LOG_FILE        = os.path.join(LOG_DIR,  "training_4b.log")
RESULTS_DIR     = os.path.join(BASE_DIR, "results",     "qwen3.5_4b")

# ── 4. PATH INTEGRITY GUARD ────────────────────────────────────────────────
def _verify_paths() -> None:
    """
    Hard-assert every critical path is under _FINETUNE_ROOT.
    Raises RuntimeError with the offending path if anything is wrong.
    """
    _critical = {
        "BASE_DIR":       BASE_DIR,
        "DATA_DIR":       DATA_DIR,
        "RAW_DATA_PATH":  RAW_DATA_PATH,
        "CHECKPOINT_DIR": CHECKPOINT_DIR,
        "BEST_MODEL_DIR": BEST_MODEL_DIR,
        "ADAPTER_DIR":    ADAPTER_DIR,
        "LOG_DIR":        LOG_DIR,
        "LOG_FILE":       LOG_FILE,
        "RESULTS_DIR":    RESULTS_DIR,
    }
    for name, path in _critical.items():
        if not path.startswith(_FINETUNE_ROOT):
            raise RuntimeError(
                f"\n\n🚨 PATH GUARD VIOLATION 🚨\n"
                f"  Variable : {name}\n"
                f"  Got      : {path}\n"
                f"  Must be under: {_FINETUNE_ROOT}\n\n"
                "This would cause checkpoints / adapters to save OUTSIDE "
                "the Finetune folder (e.g. to Drive root).\n"
                "Fix: set _FINETUNE_ROOT = '/content/drive/MyDrive/Twitter/Finetune'"
            )
    print("\n✅ All paths verified under Finetune root.")

_verify_paths()

# ── 5. ROOT DEBRIS SCANNER ─────────────────────────────────────────────────
_DEBRIS_PATTERNS = ("checkpoint-", "qwen3.5_4b", "qwen3.5_9b", "qwen3.5_4b_merged")

def _scan_root_debris(drive_root: str = _drive_root) -> None:
    found =[]
    try:
        for name in sorted(os.listdir(drive_root)):
            path = os.path.join(drive_root, name)
            if not os.path.isdir(path):
                continue
            if any(name.startswith(p) or name == p for p in _DEBRIS_PATTERNS):
                try:
                    _sz_mb = sum(
                        os.path.getsize(os.path.join(path, f))
                        for f in os.listdir(path)
                        if os.path.isfile(os.path.join(path, f))
                    ) / (1024 ** 2)
                except Exception:
                    _sz_mb = 0.0
                found.append((name, path, _sz_mb))
    except Exception as _e:
        print(f"⚠️  Could not scan Drive root: {_e}")
        return

    if not found:
        print("✅ No debris found at Drive root — clean.\n")
        return

    print("⚠️  ROOT-LEVEL DEBRIS DETECTED — these do NOT belong at Drive root:")
    print("   Delete them manually via Google Drive UI to free quota.")
    for name, path, sz in found:
        print(f"   🗑  {path}  (~{sz:.1f} MB)")
    print()

_scan_root_debris()

# ── 6. CREATE DIRECTORIES ──────────────────────────────────────────────────
for _d in[
    os.path.join(DATA_DIR, "raw"),
    os.path.join(DATA_DIR, "processed"),
    CHECKPOINT_DIR, BEST_MODEL_DIR,
    ADAPTER_DIR, LOG_DIR, RESULTS_DIR,
]:
    os.makedirs(_d, exist_ok=True)

# ── 7. LOGGING SETUP ───────────────────────────────────────────────────────
_log_fmt = "%(asctime)s | %(levelname)s | %(message)s"
logging.basicConfig(level=logging.INFO, format=_log_fmt)
logger = logging.getLogger("dei_finetune")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
_fh.setLevel(logging.INFO)
_fh.setFormatter(logging.Formatter(_log_fmt))
logger.addHandler(_fh)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO)
_sh.setFormatter(logging.Formatter(_log_fmt))
logger.addHandler(_sh)
logger.propagate = False

# ── 8. GLOBAL RANDOM SEEDS ─────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
logger.info(f"Seeds fixed: {SEED}")

# ── 9. CUDA VERIFICATION & HF LOGIN ────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. GPU runtime required.")
_gpu_name = torch.cuda.get_device_name(0)
_vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
logger.info(f"GPU: {_gpu_name} | VRAM: {_vram_gb:.1f} GB")

huggingface_hub.login(token=_hf_token, add_to_git_credential=False)

logger.info(
    f"✅ Drive mounted | ✅ Paths verified | ✅ CUDA OK | "
    f"✅ Model target: Qwen/Qwen3.5-4B"
)

====++++++====
Block 1 Output
====++++++====

⏳ Step 1: Checking permissions (Drive & HuggingFace)...
Mounted at /content/drive
✅ HF_TOKEN retrieved automatically from Colab Secrets.

⏳ Installing / upgrading packages — this takes ~5 min...
   (You have already granted permissions, so you can safely step away!)
✅ Packages updated. Refreshing sys.path to continue without stopping...

✅ All paths verified under Finetune root.
⚠️  ROOT-LEVEL DEBRIS DETECTED — these do NOT belong at Drive root:
   Delete them manually via Google Drive UI to free quota.
   🗑  /content/drive/MyDrive/checkpoint-115  (~0.0 MB)
   🗑  /content/drive/MyDrive/checkpoint-230  (~0.0 MB)
   🗑  /content/drive/MyDrive/checkpoint-345  (~80.5 MB)

2026-05-01 06:07:43,099 | INFO | Seeds fixed: 42
2026-05-01 06:07:43,624 | INFO | GPU: Tesla T4 | VRAM: 14.6 GB


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2026-05-01 06:07:45,074 | INFO | ✅ Drive mounted | ✅ Paths verified | ✅ CUDA OK | ✅ Model target: Qwen/Qwen3.5-4B


In [ ]:
# ==========================
# Block 1.5: Single-File Data Load, Stratified Split & Class Weights
# Version: v2.0
# Purpose: Load merged_4k_v2_train.csv, perform a reproducible stratified 80/20
#          train/val split in memory, validate data integrity, and compute class weights.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (Single File Load ✅): Loads only merged_4k_v2_train.csv. There is no
#   separate val file in this pipeline version. The entire split logic lives here in
#   Block 1.5, keeping data handling self-contained. This is a deliberate architectural
#   choice: the upstream data pipeline no longer needs to know about train/val splits,
#   and we can adjust the split ratio by changing VAL_SPLIT_RATIO in one place.
#
# - Feature 2 (Defensive Column Rename ✅): Attempts a rename from legacy column names
#   "text" → "Tweet_clean" and "label" → "DEI". If the CSV already has the correct
#   column names (as merged_4k_v2_train.csv does), rename() is a no-op. This guard
#   ensures the block works correctly whether run on the current file or any previous
#   pipeline output without code changes.
#
# - Feature 3 (Explicit NaN Drop ✅): Calls dropna(subset=["Tweet_clean","DEI"]) and
#   logs exact counts before and after. This was a critical bug in v1.0 which silently
#   dropped 54 rows via astype(int) coercion. v2.0 makes all data loss explicit and
#   logged, satisfying the journal contract that every transformation must be auditable.
#
# - Feature 4 (Stratified 80/20 Split ✅): Uses sklearn's train_test_split with
#   stratify=df["DEI"] and random_state=SEED. Stratification guarantees that both
#   train and val sets have the same class ratio as the full dataset. Without
#   stratification, a random split on a ~5k dataset could accidentally put all
#   hard examples in one split. The 80/20 ratio gives ~4000 train / ~1000 val, which
#   is sufficient for both stable gradient updates and reliable F1 estimation.
#   The split is done AFTER NaN removal so start_idx counts are consistent.
#
# - Feature 5 (Integer Cast ✅): Forces DEI column to int after NaN removal on both
#   train_df and val_df. Float labels (0.0, 1.0) cause label_token_ids lookup to fail
#   silently in Block 4's compute_loss — the cast prevents this.
#
# - Feature 6 (Column Validation ✅): Hard-asserts Tweet_clean and DEI exist in both
#   splits after all transforms. Raises RuntimeError with the actual column names found
#   if either is missing, rather than producing a confusing KeyError later.
#
# - Feature 7 (Class Weights ✅): Recomputes sklearn balanced class weights on
#   train_df only (not val_df). Using val labels to compute weights would constitute
#   data leakage. Weights are clipped to [0.0, 5.0] and cast to float32 PyTorch tensor
#   for injection into DEITrainer in Block 4.
#
# - Feature 8 (Distribution Logging ✅): Logs label distribution for both train and val
#   splits separately. If the stratified split worked correctly, both distributions
#   should mirror the full dataset ratio within ±1 count due to integer rounding.
#
# How it works:
# - Step 1: Load single CSV with pd.read_csv.
# - Step 2: Defensive rename for legacy compatibility.
# - Step 3: Hard-assert required columns exist.
# - Step 4: dropna on Tweet_clean and DEI, log dropped count.
# - Step 5: Cast DEI to int.
# - Step 6: Stratified 80/20 split via train_test_split.
# - Step 7: Log distribution of both splits.
# - Step 8: Compute and log class weights tensor on train split.
#
# Solutions attempted that did not work:
# - v1.0: Used separate train/val CSV files — coupled the data pipeline to the training
#   pipeline. If the upstream pipeline changed the split ratio, the training pipeline
#   had to be updated in two places. Fixed in v2.0 by doing the split here.
# - v1.0: random_state not set in train_test_split — different runs produced different
#   splits, making F1 comparisons across sessions unreliable. Fixed by using SEED.
#
# Solutions implemented:
# - v2.0: In-memory stratified split with fixed seed. Single file input.
#   All data transforms (NaN drop, cast, split) logged with before/after counts.

# ==========================
# Block 1.5: Single-File Data Load, Hard-Balanced Sampling & Stratified Split
# Version: v3.0
# Changes from v2.1:
#   - Replaced weighted random sampling with hard greedy category balancing.
#     Algorithm: sort categories by size ascending, greedily take all from
#     small categories, redistribute remaining budget equally across larger ones.
#     This guarantees full representation of small categories.
#   - Verbose per-category logging for both DEI=0 and DEI=1.
# ==========================

print("====+++++++++====")
print("Block 1.5 Output")
print("====+++++++++====")
print()

import pandas as pd
import numpy as np
import torch
import logging
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

logger = logging.getLogger("dei_finetune")
logger.info("--- Starting Block 1.5 v3.0: Single-File Load + Hard Category Balance + Stratified Split ---")

VAL_SPLIT_RATIO = 0.20

# ── Load single CSV ────────────────────────────────────────────────────────
logger.info(f"Loading dataset from: {RAW_DATA_PATH}")
full_df = pd.read_csv(RAW_DATA_PATH)
logger.info(f"Raw shape: {full_df.shape} | columns: {list(full_df.columns)}")

# ── Defensive rename ───────────────────────────────────────────────────────
full_df = full_df.rename(columns={
    "text": "Tweet_clean", "Tweet": "Tweet_clean", "label": "DEI"
})
logger.info(f"Post-rename columns: {list(full_df.columns)}")

# ── Hard-assert required columns ──────────────────────────────────────────
for _col in ["Tweet_clean", "DEI"]:
    if _col not in full_df.columns:
        raise RuntimeError(
            f"FATAL: Required column '{_col}' not found. "
            f"Actual columns: {list(full_df.columns)}"
        )
logger.info("✅ Column presence assertion passed.")

# ── Explicit NaN drop ──────────────────────────────────────────────────────
_before = len(full_df)
full_df = full_df.dropna(subset=["Tweet_clean", "DEI"]).reset_index(drop=True)
_dropped = _before - len(full_df)
if _dropped > 0:
    logger.warning(f"Dropped {_dropped} rows with NaN. Before: {_before} | After: {len(full_df)}")
else:
    logger.info(f"✅ No NaN rows found. All {len(full_df)} rows retained.")

full_df["DEI"] = full_df["DEI"].astype(int)

# ── Verbose pre-balance distribution ──────────────────────────────────────
logger.info("=" * 60)
logger.info("PRE-BALANCE DATASET OVERVIEW")
logger.info("=" * 60)
for dei_val in [0, 1]:
    subset = full_df[full_df["DEI"] == dei_val]
    logger.info(f"DEI={dei_val}: {len(subset)} tweets total")
    if "category" in full_df.columns:
        cat_counts = subset["category"].value_counts().sort_values()
        for cat, cnt in cat_counts.items():
            pct = cnt / len(subset) * 100
            logger.info(f"    {cat:<40s}: {cnt:>5} tweets  ({pct:.1f}%)")
    else:
        logger.warning("  No 'category' column found — skipping category breakdown.")
logger.info("=" * 60)


# ── Hard greedy category balancer ─────────────────────────────────────────
def hard_balance_by_category(
    df_dei: pd.DataFrame,
    target_n: int,
    seed: int,
    dei_label: int,
) -> pd.DataFrame:
    """
    Greedily samples `target_n` rows from df_dei using hard per-category
    balancing.

    Algorithm (iterative, smallest-first):
      1. Sort categories by size ascending.
      2. For each category (smallest first):
           per_cat_quota = remaining_budget // remaining_category_count
           if category_size <= per_cat_quota → take ALL rows from this category
           else                              → take exactly per_cat_quota rows
      3. Repeat until all categories are processed.

    This guarantees:
      - Small categories are fully included (no data waste).
      - The budget is redistributed fairly after each exhausted category.
      - Total output == target_n (or less if the dataset itself is smaller).
    """
    logger.info(f"  Hard balancing DEI={dei_label} | target={target_n}")

    if "category" not in df_dei.columns:
        logger.warning(f"  No 'category' column — falling back to random sample.")
        n = min(target_n, len(df_dei))
        return df_dei.sample(n=n, random_state=seed).reset_index(drop=True)

    # Sort categories by ascending size
    cat_sizes = df_dei["category"].value_counts().sort_values()
    n_cats    = len(cat_sizes)
    logger.info(f"  Found {n_cats} categories:")
    for cat, cnt in cat_sizes.items():
        logger.info(f"    '{cat}': {cnt} tweets")

    selected          = []
    remaining_budget  = target_n
    remaining_cats    = list(cat_sizes.index)   # already sorted asc

    for cat in list(cat_sizes.index):
        n_remaining_cats = len(remaining_cats)
        per_cat_quota    = remaining_budget // n_remaining_cats
        cat_df           = df_dei[df_dei["category"] == cat]

        if len(cat_df) <= per_cat_quota:
            # Take everything — category is too small to fill its quota
            taken = cat_df
            logger.info(
                f"    '{cat}': TAKE ALL {len(taken)} "
                f"(size {len(cat_df)} ≤ quota {per_cat_quota}) | "
                f"remaining budget after: {remaining_budget - len(taken)}"
            )
        else:
            taken = cat_df.sample(n=per_cat_quota, random_state=seed)
            logger.info(
                f"    '{cat}': take {len(taken)} of {len(cat_df)} "
                f"(quota={per_cat_quota}) | "
                f"remaining budget after: {remaining_budget - len(taken)}"
            )

        selected.append(taken)
        remaining_budget -= len(taken)
        remaining_cats.remove(cat)

        if remaining_budget <= 0:
            logger.info(f"    Budget exhausted — stopping early.")
            break

    result = pd.concat(selected).reset_index(drop=True)

    # Verbose post-balance summary
    logger.info(f"  Post-balance DEI={dei_label}: {len(result)} rows selected (target was {target_n})")
    final_cat_counts = result["category"].value_counts().sort_values()
    for cat, cnt in final_cat_counts.items():
        pct = cnt / len(result) * 100
        logger.info(f"    '{cat}': {cnt:>5} tweets  ({pct:.1f}%)")

    return result


# ── Apply balancing ────────────────────────────────────────────────────────
logger.info("=" * 60)
logger.info("APPLYING HARD CATEGORY BALANCE")
logger.info("=" * 60)

dei_counts  = full_df["DEI"].value_counts()
target_per_dei = int(dei_counts.min())
logger.info(
    f"DEI=0 count: {dei_counts.get(0, 0)} | "
    f"DEI=1 count: {dei_counts.get(1, 0)} | "
    f"Target per class: {target_per_dei}"
)

balanced_dfs = []
for dei_val in [0, 1]:
    df_dei = full_df[full_df["DEI"] == dei_val].copy()
    df_balanced = hard_balance_by_category(
        df_dei,
        target_n=target_per_dei,
        seed=SEED,
        dei_label=dei_val,
    )
    balanced_dfs.append(df_balanced)

full_df = pd.concat(balanced_dfs).reset_index(drop=True)

logger.info("=" * 60)
logger.info("POST-BALANCE FINAL OVERVIEW")
logger.info("=" * 60)
for dei_val in [0, 1]:
    subset = full_df[full_df["DEI"] == dei_val]
    logger.info(f"DEI={dei_val}: {len(subset)} tweets")
    if "category" in full_df.columns:
        for cat, cnt in subset["category"].value_counts().sort_values().items():
            pct = cnt / len(subset) * 100
            logger.info(f"    {cat:<40s}: {cnt:>5}  ({pct:.1f}%)")
logger.info(f"Total balanced dataset: {len(full_df)} rows")
logger.info("=" * 60)

# ── Stratified 80/20 split ────────────────────────────────────────────────
logger.info(f"Performing stratified split — train: 80% | val: 20% | seed: {SEED}")
train_df, val_df = train_test_split(
    full_df,
    test_size=VAL_SPLIT_RATIO,
    stratify=full_df["DEI"],
    random_state=SEED,
    shuffle=True,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

_train_dist = train_df["DEI"].value_counts().to_dict()
_val_dist   = val_df["DEI"].value_counts().to_dict()
logger.info(f"✅ Train set: {len(train_df):,} rows | distribution: {_train_dist}")
logger.info(f"✅ Val   set: {len(val_df):,} rows | distribution: {_val_dist}")

_full_pos_pct  = full_df["DEI"].value_counts(normalize=True).get(1, 0) * 100
_train_pos_pct = train_df["DEI"].value_counts(normalize=True).get(1, 0) * 100
_val_pos_pct   = val_df["DEI"].value_counts(normalize=True).get(1, 0) * 100
logger.info(
    f"DEI=1 prevalence — Full: {_full_pos_pct:.2f}% | "
    f"Train: {_train_pos_pct:.2f}% | Val: {_val_pos_pct:.2f}%"
)
if abs(_train_pos_pct - _full_pos_pct) > 2.0:
    logger.warning("Stratification may have failed — class ratio deviation > 2%.")
else:
    logger.info("✅ Stratification verified.")

# ── Class weights (train only) ─────────────────────────────────────────────
_cw = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["DEI"].values,
)
_cw = np.clip(_cw, a_min=0.0, a_max=5.0)
# class_weights_tensor = torch.tensor(_cw, dtype=torch.float32)
class_weights_tensor = torch.tensor(_cw, dtype=torch.float32)

logger.info(f"✅ Class Weights: DEI=0 → {_cw[0]:.4f} | DEI=1 → {_cw[1]:.4f}")
logger.info("✅ Block 1.5 v3.0 Complete. Ready to run Block 2.")

In [ ]:
# ==========================
# Block 2: Prompt Formatting & Dataset Construction
# Version: v5.0
# Purpose: Replace tokenizer Jinja template with a minimal clean version that
#          produces zero <think> tokens, verify label token IDs, format all
#          prompts, build HuggingFace Datasets, configure DataCollator.
# ==========================
#
# What this block does: (~430 words)
#
# - Feature 1 (Tokenizer Load ✅): Loads the Qwen3.5-4B tokenizer from HuggingFace.
#   The 4B and 9B models share the same tokenizer architecture, so all token ID
#   lookups ("0" → 15, "1" → 16) are identical. Sets pad_token = eos_token if
#   unset. Forces padding_side = "right" for SFT teacher-forcing — DataCollator
#   requires right-padding during training so the assistant label token is always
#   at the end of the sequence. Block 6 switches to left-padding locally for
#   batched generation and restores it in a finally block.
#
# - Feature 2 (Minimal Template Replacement ✅ — THE REAL FIX): The downloaded
#   Qwen3.5 chat template is a 7k-character Jinja2 behemoth containing vision
#   macros, MTP blocks, thinking conditionals, and reasoning_content variables.
#   All previous approaches to suppress <think> by patching this template failed:
#   v2.0 passed chat_template_kwargs as a dict (wrong API), v3.0 used
#   enable_thinking=False kwarg (correct API but template version ignores it),
#   v4.0 used regex surgery that left dangling endif tags inside unclosed macro
#   blocks causing TemplateSyntaxError. The correct fix is wholesale replacement:
#   discard the entire downloaded template and assign a 5-line Jinja2 string that
#   handles only what this pipeline needs. The replacement template is provably
#   correct because we wrote it and it contains zero thinking logic.
#
# - Feature 3 (Template Validation ✅): After replacement, a dry-run call to
#   apply_chat_template immediately validates Jinja2 syntax and asserts zero
#   <think> tokens in both training and inference format strings. A failure here
#   means the replacement string itself is broken — which is impossible since it is
#   a literal we control. This serves as a regression guard for future edits.
#
# - Feature 4 (Label Token Assertion ✅): Verifies that "0" and "1" each tokenize
#   to exactly one token ID. For Qwen3.5-4B: "0" → token 15, "1" → token 16.
#   DataCollatorForCompletionOnlyLM requires single-token labels. If this assertion
#   fails, the entire training pipeline produces incorrect loss — this check must
#   run before any dataset is built.
#
# - Feature 5 (Prompt Templates ✅): format_prompt() produces the full
#   system→user→assistant:label turn for SFT. format_inference_prompt() produces
#   system→user with an open assistant turn for generation. Both use
#   apply_chat_template with no special kwargs — the clean template handles it.
#
# - Feature 6 (Hard Sanity Check ✅): Asserts zero <think> in both training and
#   inference spot-check prompts before formatting the full dataset. With a template
#   we wrote ourselves, this is a guarantee, not a hope.
#
# - Feature 7 (HF Dataset Construction ✅): Converts formatted string lists into
#   HuggingFace Dataset objects from train_df and val_df produced by Block 1.5.
#   Tokenization is deferred to Block 5 to keep the dependency graph acyclic.
#
# - Feature 8 (DataCollatorForCompletionOnlyLM ✅): Response template set to
#   "<|im_start|>assistant\n" so loss is computed only on the label token.
#
# How it works:
# - Step 1: Load tokenizer for Qwen/Qwen3.5-4B.
# - Step 2: Assign _MINIMAL_CHAT_TEMPLATE to tokenizer.chat_template.
# - Step 3: Dry-run validate Jinja syntax + zero <think>.
# - Step 4: Assert label tokens are single-token.
# - Step 5: Spot-check both prompt formats, hard assert.
# - Step 6: Format full train and val datasets.
# - Step 7: Build HF Datasets, configure DataCollator.
#
# Solutions attempted that did not work:
# - v2.0: chat_template_kwargs={"enable_thinking": False} — wrong API.
# - v3.0: enable_thinking=False kwarg — correct API, template ignores it.
# - v4.0: Regex surgery on existing template — TemplateSyntaxError from
#         dangling endif inside unclosed macro block.
#
# Solutions implemented:
# - v5.0: Wholesale template replacement with a 5-line Jinja2 string.

print("====++++====")
print("Block 2 Output")
print("====++++====")
print()

import logging
import re
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM

logger = logging.getLogger("dei_finetune")
MODEL_NAME = "Qwen/Qwen3.5-4B"

# ── Load Tokenizer ─────────────────────────────────────────────────────────
logger.info(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    logger.warning(
        f"pad_token was None — set to eos_token. "
        f"pad_token_id={tokenizer.pad_token_id}"
    )
else:
    logger.info(
        f"pad_token already set: '{tokenizer.pad_token}' "
        f"(id={tokenizer.pad_token_id})"
    )

tokenizer.padding_side = "right"
logger.info("tokenizer.padding_side set to 'right' for SFT training.")

# ── Minimal Chat Template Replacement ─────────────────────────────────────
# Wholesale replacement — no patching of the existing template.
# This 5-line Jinja2 produces exactly:
#   Training:  <|im_start|>system\n...<|im_end|>\n<|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n{label}<|im_end|>\n
#   Inference: same but without the final label + <|im_end|>
_MINIMAL_CHAT_TEMPLATE = (
    "{%- for message in messages %}"
    "{{- '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "{{- '<|im_start|>assistant\n' }}"
    "{%- endif %}"
)

_original_len = len(tokenizer.chat_template) if tokenizer.chat_template else 0
tokenizer.chat_template = _MINIMAL_CHAT_TEMPLATE
logger.info(
    f"Chat template replaced — "
    f"original length: {_original_len} chars | "
    f"new length: {len(tokenizer.chat_template)} chars"
)

# ── Dry-Run Validation ─────────────────────────────────────────────────────
_DRY_SYSTEM = "You are a binary classifier. Reply with only 0 or 1."
_dry_train_check = tokenizer.apply_chat_template(
    [
        {"role": "system",    "content": _DRY_SYSTEM},
        {"role": "user",      "content": "Tweet: test tweet"},
        {"role": "assistant", "content": "1"},
    ],
    tokenize=False,
    add_generation_prompt=False,
)
_dry_infer_check = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": _DRY_SYSTEM},
        {"role": "user",   "content": "Tweet: test tweet"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
logger.info(f"Dry-run training format:\n{_dry_train_check}\n{'─'*60}")
logger.info(f"Dry-run inference format:\n{_dry_infer_check}\n{'─'*60}")

if "<think>" in _dry_train_check or "<think>" in _dry_infer_check:
    raise RuntimeError(
        "FATAL: <think> in dry-run output. Minimal replacement template "
        "should never produce thinking tokens. Check _MINIMAL_CHAT_TEMPLATE."
    )
logger.info("✅ Dry-run passed: Jinja compiles cleanly, zero <think> tokens.")

# ── System Prompt ──────────────────────────────────────────────────────────
_SYSTEM = (
    "You are a binary classifier. Your task is to identify whether a university "
    "tweet is about a Diversity, Equity, and Inclusion (DEI) initiative or program.\n\n"

    "Label 1 — DEI tweet. The tweet explicitly discusses one or more of:\n"
    "  - Structural diversity initiatives: DEI offices, diversity hiring, equity programs, "
    "affirmative action, representation goals\n"
    "  - Inclusion efforts: accessibility accommodations, disability services, "
    "anti-discrimination policies, belonging campaigns, safe space programs\n"
    "  - Identity-focused support: programs for women, LGBTQ+, first-gen students, "
    "students of color, international students, veterans, or other underrepresented groups\n"
    "  - Equity in outcomes: pay equity audits, closing achievement gaps, equitable "
    "admissions, food/housing insecurity support framed as equity work\n"
    "  - DEI training or education: implicit bias workshops, cultural competency training, "
    "diversity speaker series explicitly framed as DEI\n\n"

    "Label 0 — Not a DEI tweet. The tweet is about:\n"
    "  - General campus news, rankings, events, or announcements not framed around equity\n"
    "  - Cultural or holiday celebrations (Black History Month post, Diwali greeting, "
    "Pride flag raising) that congratulate or celebrate without discussing structural change\n"
    "  - Biomedical or social science research that studies health disparities but is not "
    "a university DEI program\n"
    "  - Standard university marketing: admissions stats, donor news, sports, alumni\n"
    "  - Individual student or faculty achievement announcements not tied to a DEI program\n"
    "  - Mentions of diversity as an incidental descriptor, not as the focus\n\n"

    "The key distinction: a tweet must be ABOUT a DEI initiative or structural effort, "
    "not merely mention diversity, feature a diverse person, or celebrate a cultural moment.\n\n"

    "Reply with only the digit 0 or 1. No explanation. No reasoning."
)

# ── Verify Label Tokens Are Single Tokens ──────────────────────────────────
for _label_str in tqdm(["0", "1"], desc="Verifying label token IDs"):
    _ids = tokenizer(_label_str, add_special_tokens=False)["input_ids"]
    if len(_ids) != 1:
        raise RuntimeError(
            f"Label string '{_label_str}' tokenised to {len(_ids)} tokens: {_ids}. "
            "DataCollatorForCompletionOnlyLM requires a single-token label."
        )

label_token_ids = {
    0: tokenizer("0", add_special_tokens=False)["input_ids"][0],
    1: tokenizer("1", add_special_tokens=False)["input_ids"][0],
}
logger.info(
    f"Label token IDs verified — '0': {label_token_ids[0]} | "
    f"'1': {label_token_ids[1]}"
)

# ── Prompt Templates ───────────────────────────────────────────────────────
def format_prompt(tweet: str, label: int) -> str:
    """SFT training prompt: system + user + assistant label token."""
    messages = [
        {"role": "system",    "content": _SYSTEM},
        {"role": "user",      "content": f"Tweet: {tweet}"},
        {"role": "assistant", "content": str(label)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )

def format_inference_prompt(tweet: str) -> str:
    """Inference prompt: system + user, open assistant turn for generation."""
    messages = [
        {"role": "system", "content": _SYSTEM},
        {"role": "user",   "content": f"Tweet: {tweet}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

# ── Spot-Check ────────────────────────────────────────────────────────────
_sample_train = format_prompt("Our diversity office is hosting a STEM workshop.", 1)
_sample_infer = format_inference_prompt("Congrats to all our graduates!")
logger.info(f"[FORMAT CHECK] Training prompt:\n{_sample_train}\n{'─'*60}")
logger.info(f"[FORMAT CHECK] Inference prompt:\n{_sample_infer}\n{'─'*60}")

if "<think>" in _sample_train:
    raise RuntimeError(f"FATAL: <think> in training prompt.")
if "<think>" in _sample_infer:
    raise RuntimeError(f"FATAL: <think> in inference prompt.")
logger.info("✅ FORMAT SANITY CHECK PASSED: Zero <think> tokens in all prompts.")

# ── Format Full Datasets ───────────────────────────────────────────────────
logger.info("Formatting training prompts...")
train_texts = [
    format_prompt(row["Tweet_clean"], row["DEI"])
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Format train")
]
logger.info("Formatting validation prompts...")
val_texts = [
    format_prompt(row["Tweet_clean"], row["DEI"])
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Format val")
]

# ── Build HuggingFace Datasets ─────────────────────────────────────────────
train_hf_dataset = Dataset.from_dict({"text": train_texts})
val_hf_dataset   = Dataset.from_dict({"text": val_texts})
logger.info(
    f"HF Datasets built — train: {len(train_hf_dataset)} | val: {len(val_hf_dataset)}"
)

# ── DataCollatorForCompletionOnlyLM ───────────────────────────────────────
_response_template_str = "<|im_start|>assistant\n"
_response_template_ids = tokenizer(
    _response_template_str, add_special_tokens=False
)["input_ids"]
logger.info(
    f"Response template: {_response_template_str!r} → "
    f"token IDs: {_response_template_ids}"
)
collator = DataCollatorForCompletionOnlyLM(
    response_template=_response_template_ids,
    tokenizer=tokenizer,
    mlm=False,
)
logger.info("DataCollatorForCompletionOnlyLM configured.")
logger.info(
    "✅ Block 2 Complete — "
    "minimal template installed | "
    "zero <think> guaranteed by construction | "
    "DataCollator ready."
)

In [ ]:
# ==========================
# Block 3: Model & Tokenizer Loading (QLoRA 4-bit, Qwen3.5-4B)
# Version: v2.0
# Purpose: Configure BitsAndBytes 4-bit quantization, load Qwen3.5-4B base model,
#          apply LoRA adapter, and run a mandatory dry-run validating zero <think>.
# ==========================
#
# What this block does: (~400 words)
#
# - Feature 1 (Memory Safety ✅): Loads BitsAndBytes config with nf4 4-bit
#   quantization and double quantization (bnb_4bit_use_double_quant=True). Double
#   quantization quantizes the quantization constants themselves, saving an additional
#   ~0.4 bits per parameter. On T4 (14.6 GB VRAM), Qwen3.5-4B in 4-bit occupies
#   ~2.5 GB, leaving ~12 GB for activations, gradients, and optimizer states —
#   substantially more headroom than the 9B model (~5 GB). compute_dtype must be
#   float16 not bfloat16: T4 is Turing architecture (sm_75) which lacks native
#   bfloat16 hardware support. Mixing bfloat16 compute with float16 GradScaler
#   causes silent gradient corruption within the first 50 steps.
#
# - Feature 2 (Model Loading ✅): Loads Qwen/Qwen3.5-4B with device_map="auto"
#   to let HuggingFace place layers optimally across available GPU memory. Sets
#   use_cache=False (gradient checkpointing requires this) and pretraining_tp=1
#   (disables tensor parallelism which is incompatible with PEFT on single GPU).
#
# - Feature 3 (Generation Config Neutralization ✅): Resets do_sample=False,
#   temperature=1.0, top_p=1.0, top_k=0. The model's default generation config
#   has sampling enabled; leaving it active during the dry-run would make output
#   non-deterministic, preventing reliable assertion of "1" as first char.
#
# - Feature 4 (LoRA Injection ✅): Applies LoRA with r=16, alpha=32, dropout=0.05
#   targeting all seven attention and MLP projection matrices. With 4B parameters,
#   r=16 gives 0.53% trainable parameters (~22M trainable vs ~4B total). This is
#   sufficient for single-task fine-tuning on ~4000 examples — higher rank would
#   risk overfitting on this dataset size. enable_input_require_grads() is called
#   explicitly to make gradient checkpointing compatible with PEFT's adapter layers.
#
# - Feature 5 (Mandatory Dry-Run ✅): Runs a single forward+generate pass to
#   confirm the model can produce output through the inference path before training
#   starts. The dry-run uses format_inference_prompt() from Block 2 and asserts the
#   prompt is clean. If the raw output contains an empty <think></think> wrapper
#   (a known Qwen3.5 tokenizer version artifact), it logs a WARNING but does not
#   raise — DataCollatorForCompletionOnlyLM masks all prompt tokens including any
#   thinking wrapper in training, so training is unaffected. strip_think_tokens()
#   handles this at inference time in Block 6.
#
# - Feature 6 (Thinking Suppression ✅): strip_think_tokens() uses compiled regex
#   to remove both complete <think>…</think> blocks and unclosed <think>…$ tails.
#   has_nonempty_thinking() distinguishes empty wrappers (acceptable) from active
#   reasoning chains (fatal for classification accuracy).
#
# How it works:
# - Step 1: Define strip_think_tokens() and has_nonempty_thinking() helpers.
# - Step 2: Build BitsAndBytesConfig.
# - Step 3: Load base model with quantization config.
# - Step 4: Neutralize generation config sampling params.
# - Step 5: Apply LoRA via get_peft_model(), call enable_input_require_grads().
# - Step 6: Log trainable parameter count and percentage.
# - Step 7: Run dry-run inference, assert output parseable.
#
# Solutions attempted that did not work:
# - Trying to set generation_config.enable_thinking = False — attribute does not
#   exist on all transformers versions; suppression via chat template is primary.
#
# Solutions implemented:
# - strip_think_tokens() as fallback safety net for evaluation in Block 6.
# - Empty <think></think> wrapper treated as WARNING not ERROR.

print("====++++++====")
print("Block 3 Output")
print("====++++++====")
print()

import logging
import re
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

logger = logging.getLogger("dei_finetune")

# ── strip_think_tokens — defensive safety net for Block 6 ─────────────────
_RE_THINK_BLOCK   = re.compile(r"<think>.*?</think>", re.DOTALL)
_RE_THINK_CONTENT = re.compile(r"<think>(.*?)</think>", re.DOTALL)

def strip_think_tokens(text: str) -> str:
    text = _RE_THINK_BLOCK.sub("", text)
    text = re.sub(r"<think>.*$", "", text, flags=re.DOTALL)
    return text.strip()

def has_nonempty_thinking(text: str) -> bool:
    matches = _RE_THINK_CONTENT.findall(text)
    return any(m.strip() for m in matches)

logger.info("strip_think_tokens() and has_nonempty_thinking() defined.")

# ── BitsAndBytesConfig ─────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 is Turing: NO native bf16
)
logger.info(
    f"BnB config — quant_type: nf4 | double_quant: True | "
    f"compute_dtype: {bnb_config.bnb_4bit_compute_dtype}"
)

# ── Load Base Model ────────────────────────────────────────────────────────
logger.info(f"Loading base model: {MODEL_NAME}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
base_model.config.use_cache       = False
base_model.config.pretraining_tp  = 1
logger.info("Base model loaded successfully.")

# ── Neutralize generation config ──────────────────────────────────────────
base_model.generation_config.do_sample   = False
base_model.generation_config.temperature = 1.0
base_model.generation_config.top_p       = 1.0
base_model.generation_config.top_k       = 0
logger.info("generation_config sampling params neutralised.")

# ── LoRA Configuration ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.15,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
logger.info(
    f"LoRA config — r={lora_config.r} | alpha={lora_config.lora_alpha} | "
    f"dropout={lora_config.lora_dropout} | "
    f"target_modules={lora_config.target_modules}"
)

model = get_peft_model(base_model, lora_config)
model.enable_input_require_grads()
logger.info(
    "enable_input_require_grads() called — QLoRA + gradient checkpointing safe."
)

_total_params     = sum(p.numel() for p in model.parameters())
_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(
    f"Model parameters — trainable: {_trainable_params:,} "
    f"({100 * _trainable_params / _total_params:.2f}%) | "
    f"total: {_total_params:,}"
)

# ── Mandatory Dry-Run Validation ───────────────────────────────────────────
logger.info("Running mandatory dry-run validation...")
_dry_tweet  = "Our company has a strong commitment to diversity and inclusion."
_dry_prompt = format_inference_prompt(_dry_tweet)

if "<think>" in _dry_prompt:
    raise RuntimeError(
        "FATAL: <think> found in dry-run prompt. "
        "format_inference_prompt() did not suppress thinking."
    )
logger.info("✅ Dry-run prompt confirmed clean (no <think> in prompt string).")

_dry_inputs = tokenizer(_dry_prompt, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    _dry_out = model.generate(
        **_dry_inputs,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
model.train()

_input_len     = _dry_inputs["input_ids"].shape[1]
_dry_generated = tokenizer.decode(
    _dry_out[0][_input_len:], skip_special_tokens=False
)
logger.info(f"Dry-run raw output: {_dry_generated!r}")

# ── HARD Assertion: no active thinking ────────────────────────────────────
if "<think>" in _dry_generated:
    if has_nonempty_thinking(_dry_generated):
        raise RuntimeError(
            f"FATAL DRY-RUN FAILURE: Active (non-empty) <think> block detected.\n"
            f"Raw output: {_dry_generated!r}\n"
            "enable_thinking suppression FAILED. Check tokenizer version."
        )
    else:
        logger.warning(
            "Dry-run: empty <think>\\n\\n</think> wrapper detected despite "
            "minimal template. This is a tokenizer version artifact. "
            "strip_think_tokens() handles this at inference. Training unaffected."
        )
else:
    logger.info("✅ Dry-run CONFIRMED: Zero <think> tokens. Suppression working.")

_dry_stripped = strip_think_tokens(_dry_generated)
logger.info(f"Dry-run stripped output: {_dry_stripped!r}")

if not _dry_stripped or _dry_stripped[0] not in ("0", "1"):
    logger.warning(
        f"Dry-run: output not yet parseable as 0 or 1 (expected for untrained model). "
        f"SFT will train this behavior."
    )
else:
    logger.info(
        f"✅ Dry-run bonus: output already parseable as '{_dry_stripped[0]}'."
    )

logger.info(
    f"✅ Block 3 Complete — no active reasoning | "
    f"stripped output: {_dry_stripped!r} | "
    f"first char: '{_dry_stripped[:1] if _dry_stripped else 'N/A'}'"
)

In [ ]:
# ==========================
# Block 4: Training Configuration & Custom Trainer Setup
# Version: v1.3
# Purpose: Assert dtype consistency, define metric functions, define
#          TrainingMonitorCallback, define DEITrainer with weighted loss,
#          atomic checkpointing, AND a save_model() override that blocks
#          unauthorized saves, then instantiate TrainingArguments.
# ==========================
#
# What this block does: (~450 words)
#
# - Feature 1 (Dtype Assertion ✅): Unchanged from v1.2. Asserts
#   bnb_4bit_compute_dtype == torch.float16 before any training object is
#   constructed to prevent silent gradient corruption on T4.
#
# - Feature 2 (preprocess_logits_for_metrics ✅): Unchanged from v1.2.
#   Extracts the predicted token ID at the label position via causal LM shift.
#
# - Feature 3 (make_compute_metrics ✅): Unchanged from v1.2. Closure factory
#   mapping predicted token IDs to F1 macro + accuracy.
#
# - Feature 4 (TrainingMonitorCallback ✅): Unchanged from v1.2.
#   on_train_begin restores best_f1 from log_history on resume.
#   on_evaluate saves best model, implements early stopping.
#   on_train_end prints full epoch summary Rich table.
#
# - Feature 5 (DEITrainer.compute_loss ✅): Unchanged from v1.2.
#   Per-sample class-weighted cross-entropy on completion token only.
#
# - Feature 6 (DEITrainer._save_checkpoint ✅): Unchanged from v1.2.
#   Atomic save to checkpoint-N.tmp → rename to checkpoint-N, rotation to
#   keep last 3 step checkpoints.
#
# - Feature 7 (DEITrainer.save_model ✅ — NEW in v1.3): This is the critical
#   fix for root-level spam. The base HuggingFace Trainer calls save_model()
#   at the end of training (and in some resume scenarios) to write the final
#   model directly to self.args.output_dir. If we don't override it, the
#   PeftModel adapter files get written to CHECKPOINT_DIR itself (not a
#   checkpoint-N subdirectory), AND if output_dir is wrong they end up at
#   Drive root. The override: (1) resolves the target path to absolute form,
#   (2) checks it starts with _FINETUNE_ROOT, (3) if unauthorized, logs a
#   WARNING and silently returns — the best model is already saved by
#   TrainingMonitorCallback.on_evaluate so blocking this duplicate save is
#   safe, (4) if authorized, delegates to model.save_pretrained + tokenizer
#   save. This intercepts both the end-of-training save and any internal
#   trl SFTTrainer save_model calls.
#
# - Feature 8 (TrainingArguments ✅): save_strategy="steps", save_steps=50,
#   save_total_limit=3. eval_strategy="epoch". fp16=True mandatory for T4.
#   Explicit output_dir set from CHECKPOINT_DIR which is now path-guarded
#   by Block 1's _verify_paths().
#
# How it works:
# - Step 1: Assert dtype consistency.
# - Step 2: Define preprocess_logits_for_metrics.
# - Step 3: Define make_compute_metrics closure factory.
# - Step 4: Define TrainingMonitorCallback (on_train_begin, on_evaluate,
#           on_train_end).
# - Step 5: Define DEITrainer (compute_loss, _save_checkpoint, save_model).
# - Step 6: Instantiate TrainingArguments.
# - Step 7: Assert fp16 flag.
#
# Solutions attempted that did not work:
# - v1.2: No save_model override — Trainer's end-of-training call to
#   save_model(output_dir) bypassed our _save_checkpoint logic entirely and
#   wrote adapter files directly to output_dir. If output_dir was ever wrong
#   (prior session with different BASE_DIR), this created root-level debris.
#   Fixed in v1.3 by adding save_model() override with path guard.
# - v1.2: No path guard in _save_checkpoint — if self.args.output_dir was
#   wrong, checkpoint-N dirs appeared at Drive root. Fixed by Block 1's
#   _verify_paths() ensuring output_dir is always correct before Block 4 runs.
#
# Solutions implemented:
# - save_model() override with _FINETUNE_ROOT authorization check.
# - All saves (checkpoint-N and final) now funnel through guarded methods.
# ==========================

print("====++++++====")
print("Block 4 Output")
print("====++++++====")
print()

import json, logging, os, random, shutil
import numpy as np, torch, torch.nn as nn
from rich.console import Console
from rich.table   import Table
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    TrainerCallback, TrainerControl, TrainerState, TrainingArguments,
)
from trl import SFTTrainer

logger = logging.getLogger("dei_finetune")
_rc    = Console()

# ── Dtype Consistency Assertion ────────────────────────────────────────────
assert bnb_config.bnb_4bit_compute_dtype == torch.float16, (
    "FATAL: BnB compute dtype must be torch.float16 on T4. "
    f"Got: {bnb_config.bnb_4bit_compute_dtype}."
)
logger.info("✅ Dtype assertion passed: bnb_4bit_compute_dtype == torch.float16")

# ── preprocess_logits_for_metrics ──────────────────────────────────────────
def preprocess_logits_for_metrics(
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    batch_size   = shift_logits.shape[0]
    pred_tokens  = torch.zeros(batch_size, dtype=torch.long, device=logits.device)
    for i in range(batch_size):
        valid_pos = (shift_labels[i] != -100).nonzero(as_tuple=True)[0]
        if len(valid_pos) > 0:
            pred_tokens[i] = shift_logits[i, valid_pos[0].item()].argmax()
    return pred_tokens

# ── make_compute_metrics ───────────────────────────────────────────────────
def make_compute_metrics(lbl_token_ids: dict):
    t0 = lbl_token_ids[0]
    t1 = lbl_token_ids[1]
    def compute_metrics(eval_pred):
        pred_tokens  = eval_pred.predictions
        label_matrix = eval_pred.label_ids
        shift_labels = label_matrix[:, 1:]
        true_classes: list[int] = []
        pred_classes: list[int] = []
        for i in range(len(pred_tokens)):
            valid_pos = np.where(shift_labels[i] != -100)[0]
            if len(valid_pos) == 0:
                continue
            true_tok = int(shift_labels[i][valid_pos[0]])
            true_cls = 0 if true_tok == t0 else (1 if true_tok == t1 else None)
            if true_cls is None:
                continue
            pred_tok = int(pred_tokens[i])
            pred_cls = 0 if pred_tok == t0 else (1 if pred_tok == t1 else 1 - true_cls)
            true_classes.append(true_cls)
            pred_classes.append(pred_cls)
        if not true_classes:
            return {"eval_f1_macro": 0.0, "eval_accuracy": 0.0}
        f1  = f1_score(true_classes, pred_classes, average="macro", zero_division=0)
        acc = accuracy_score(true_classes, pred_classes)
        return {"eval_f1_macro": round(f1, 6), "eval_accuracy": round(acc, 6)}
    return compute_metrics

compute_metrics_fn = make_compute_metrics(label_token_ids)

# ── TrainingMonitorCallback ────────────────────────────────────────────────
class TrainingMonitorCallback(TrainerCallback):
    def __init__(self, best_model_dir: str, patience: int = 2,
                 threshold: float = 0.001):
        self.best_model_dir   = best_model_dir
        self.patience         = patience
        self.threshold        = threshold
        self.best_f1          = 0.0
        self.patience_counter = 0

    def on_train_begin(self, args, state, control, **kwargs):
        prior_evals = [e for e in state.log_history if "eval_f1_macro" in e]
        if not prior_evals:
            logger.info("on_train_begin: fresh run — no prior eval history.")
            return control
        prior_best = max(e["eval_f1_macro"] for e in prior_evals)
        if prior_best > self.best_f1:
            self.best_f1          = prior_best
            self.patience_counter = 0
            logger.info(
                f"on_train_begin: restored best_f1={self.best_f1:.4f} "
                f"from {len(prior_evals)} prior eval entries."
            )
        for e in prior_evals:
            logger.info(
                f"  [RESTORED] Epoch {int(e.get('epoch',0))} | "
                f"Step {e.get('step','?')} | F1={e.get('eval_f1_macro',0):.4f}"
            )
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control
        current_f1 = metrics.get("eval_f1_macro", 0.0)
        epoch      = int(state.epoch) if state.epoch is not None else 0
        logger.info(
            f"Epoch {epoch}/{int(args.num_train_epochs)} | "
            f"Step {state.global_step} | "
            f"eval_f1_macro: {current_f1:.4f} | Best: {self.best_f1:.4f} | "
            f"Patience: {self.patience_counter}/{self.patience}"
        )
        if current_f1 > self.best_f1 + self.threshold:
            self.best_f1          = current_f1
            self.patience_counter = 0
            _m = kwargs.get("model")
            if _m is not None:
                os.makedirs(self.best_model_dir, exist_ok=True)
                _m.save_pretrained(self.best_model_dir)
                _tok = kwargs.get("tokenizer")
                if _tok is not None:
                    _tok.save_pretrained(self.best_model_dir)
                logger.info(
                    f"✅ New best model saved: F1={current_f1:.4f} → "
                    f"{self.best_model_dir}"
                )
        else:
            self.patience_counter += 1
            logger.warning(
                f"F1 did not improve. Patience: {self.patience_counter}/{self.patience}"
            )
            if self.patience_counter >= self.patience:
                control.should_training_stop = True
                logger.info("🛑 Early stopping triggered.")
        return control

    def on_train_end(self, args, state, control, **kwargs):
        eval_entries = [e for e in state.log_history if "eval_f1_macro" in e]
        if not eval_entries:
            logger.warning("on_train_end: no eval entries.")
            return control
        best_f1_seen = max(e.get("eval_f1_macro", 0.0) for e in eval_entries)
        tbl = Table(
            title="Complete Training Summary — All Epochs",
            show_header=True, header_style="bold magenta",
        )
        tbl.add_column("Epoch",     style="cyan",        justify="right", min_width=6)
        tbl.add_column("Step",                           justify="right", min_width=6)
        tbl.add_column("F1 Macro",  style="bold green",  justify="right", min_width=9)
        tbl.add_column("Accuracy",                       justify="right", min_width=9)
        tbl.add_column("Val Loss",                       justify="right", min_width=9)
        tbl.add_column("Best",      style="bold yellow", justify="center",min_width=5)
        train_loss_map: dict[int, float] = {}
        for e in state.log_history:
            if "loss" in e and "eval_f1_macro" not in e:
                train_loss_map[int(e.get("step", 0))] = e["loss"]
        for e in eval_entries:
            ep       = int(e.get("epoch", 0))
            step     = int(e.get("step", 0))
            f1       = e.get("eval_f1_macro", 0.0)
            acc      = e.get("eval_accuracy", 0.0)
            val_loss = e.get("eval_loss", float("nan"))
            is_best  = abs(f1 - best_f1_seen) < 1e-6
            tbl.add_row(
                str(ep), str(step), f"{f1:.4f}",
                f"{acc:.4f}" if acc else "—",
                f"{val_loss:.4f}" if val_loss == val_loss else "—",
                "★" if is_best else "",
            )
        _rc.print(tbl)
        logger.info(f"on_train_end: best F1 across all epochs: {best_f1_seen:.4f}")
        return control


# ── DEITrainer ─────────────────────────────────────────────────────────────
class DEITrainer(SFTTrainer):
    def __init__(self, *args, class_weights=None, label_token_ids_map=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is None:
            raise ValueError("DEITrainer requires class_weights tensor.")
        if label_token_ids_map is None:
            raise ValueError("DEITrainer requires label_token_ids_map dict.")
        self.cw      = class_weights
        self.lbl_ids = label_token_ids_map
        self.best_f1 = 0.0

    # ── compute_loss ──────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False,
                     num_items_in_batch=None):
        labels       = inputs.get("labels").clone()
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs      = model(**model_inputs)
        logits       = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        B, S, V      = shift_logits.shape
        loss_fct       = nn.CrossEntropyLoss(ignore_index=-100, reduction="none")
        per_token_loss = loss_fct(
            shift_logits.reshape(-1, V), shift_labels.reshape(-1),
        ).reshape(B, S)
        valid_mask      = (shift_labels != -100).float()
        valid_count     = valid_mask.sum(dim=-1).clamp(min=1e-8)
        per_sample_loss = (per_token_loss * valid_mask).sum(dim=-1) / valid_count
        cw             = self.cw.to(shift_logits.device)
        sample_weights = torch.ones(B, device=shift_logits.device, dtype=torch.float32)
        tok0, tok1     = self.lbl_ids[0], self.lbl_ids[1]
        for i in range(B):
            valid_pos = (shift_labels[i] != -100).nonzero(as_tuple=True)[0]
            if len(valid_pos) > 0:
                true_tok = shift_labels[i][valid_pos[0]].item()
                if true_tok == tok0:
                    sample_weights[i] = cw[0]
                elif true_tok == tok1:
                    sample_weights[i] = cw[1]
        weighted_loss = (per_sample_loss * sample_weights).mean()
        return (weighted_loss, outputs) if return_outputs else weighted_loss

    # ── _save_checkpoint ──────────────────────────────────────────────────
    def _save_checkpoint(self, model, trial, metrics=None):
        step       = self.state.global_step
        run_dir    = self.args.output_dir
        # ── GUARD: ensure run_dir is under Finetune root ───────────────────
        if not os.path.abspath(run_dir).startswith(_FINETUNE_ROOT):
            raise RuntimeError(
                f"_save_checkpoint BLOCKED: output_dir='{run_dir}' is outside "
                f"_FINETUNE_ROOT='{_FINETUNE_ROOT}'. "
                "This would spam Drive root. Fix Block 1 path constants."
            )
        output_dir = os.path.join(run_dir, f"checkpoint-{step}")
        tmp_dir    = output_dir + ".tmp"
        if os.path.exists(tmp_dir):
            shutil.rmtree(tmp_dir, ignore_errors=True)
        os.makedirs(tmp_dir, exist_ok=True)
        model.save_pretrained(tmp_dir)
        _tokenizer = getattr(self, "processing_class", getattr(self, "tokenizer", None))
        if _tokenizer:
            _tokenizer.save_pretrained(tmp_dir)
        if self.optimizer is not None:
            try:
                torch.save(self.optimizer.state_dict(),
                           os.path.join(tmp_dir, "optimizer.pt"))
            except Exception as _e:
                logger.warning(f"Could not save optimizer state: {_e}")
        if self.lr_scheduler is not None:
            try:
                torch.save(self.lr_scheduler.state_dict(),
                           os.path.join(tmp_dir, "scheduler.pt"))
            except Exception as _e:
                logger.warning(f"Could not save scheduler state: {_e}")
        _rng_states = {
            "python": random.getstate(),
            "numpy":  np.random.get_state(),
            "cpu":    torch.random.get_rng_state(),
            "cuda":   torch.cuda.random.get_rng_state_all() if torch.cuda.is_available() else [],
        }
        torch.save(_rng_states, os.path.join(tmp_dir, "rng_state_0.pth"))
        self.state.save_to_json(os.path.join(tmp_dir, "trainer_state.json"))
        _cb   = getattr(self, "_monitor_callback_ref", None)
        _best = _cb.best_f1 if _cb is not None else self.best_f1
        torch.save(
            {"best_f1": _best, "epoch": self.state.epoch,
             "global_step": self.state.global_step},
            os.path.join(tmp_dir, "custom_state.pt"),
        )
        # Atomic swap
        if os.path.exists(output_dir):
            _old_tmp = output_dir + ".old"
            os.rename(output_dir, _old_tmp)
            os.rename(tmp_dir, output_dir)
            shutil.rmtree(_old_tmp, ignore_errors=True)
        else:
            os.rename(tmp_dir, output_dir)
        logger.info(f"Atomic checkpoint saved: {output_dir}")
        # Rotate — keep last 3 step checkpoints
        _ckpts = sorted(
            [d for d in os.listdir(run_dir)
             if d.startswith("checkpoint-") and not d.endswith((".tmp", ".old"))
             and d.split("-")[-1].isdigit()],
            key=lambda x: int(x.split("-")[-1]),
        )
        while len(_ckpts) > 3:
            _old = os.path.join(run_dir, _ckpts.pop(0))
            shutil.rmtree(_old, ignore_errors=True)
            logger.info(f"Rotated out: {_old}")

    # ── save_model (NEW v1.3) ─────────────────────────────────────────────
    def save_model(self, output_dir: str | None = None, _internal_call: bool = False):
        """
        Intercept ALL save_model() calls — including HuggingFace Trainer's
        end-of-training call and trl SFTTrainer's internal calls.

        Authorization logic:
          - Resolve target to absolute path.
          - If it starts with _FINETUNE_ROOT → allow, save model + tokenizer.
          - If it does NOT start with _FINETUNE_ROOT → BLOCK the save and
            log a WARNING. Best model is already saved by on_evaluate so
            blocking this duplicate is safe and prevents root spam.

        This is the root cause fix: without this override, Trainer.train()
        calls self.save_model(self.args.output_dir) after the loop ends,
        and SFTTrainer may call it during initialization. Both calls bypass
        _save_checkpoint and write directly to whatever output_dir resolves to.
        """
        target = os.path.abspath(output_dir) if output_dir else os.path.abspath(
            self.args.output_dir
        )
        if not target.startswith(_FINETUNE_ROOT):
            logger.warning(
                f"save_model() BLOCKED — unauthorized target: '{target}'. "
                f"Target must be under '{_FINETUNE_ROOT}'. "
                "Best model is already saved by TrainingMonitorCallback.on_evaluate."
            )
            return  # Do NOT save — prevents root spam
        logger.info(f"save_model() → authorized, saving to: {target}")
        os.makedirs(target, exist_ok=True)
        self.model.save_pretrained(target)
        _tok = getattr(self, "processing_class", getattr(self, "tokenizer", None))
        if _tok is not None:
            _tok.save_pretrained(target)
        logger.info(f"✅ save_model() complete: {target}")


# ── TrainingArguments ──────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,        # guarded by Block 1
    num_train_epochs            = 5,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,
    per_device_eval_batch_size  = 8,
    fp16                        = True,
    bf16                        = False,
    optim                       = "paged_adamw_8bit",
    gradient_checkpointing      = True,
    learning_rate               = 5e-4,
    weight_decay                = 0.05,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.10,
    eval_strategy               = "epoch",
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 3,
    logging_steps               = 25,
    load_best_model_at_end      = False,
    metric_for_best_model       = "eval_f1_macro",
    greater_is_better           = True,
    report_to                   = "none",
    seed                        = SEED,
    dataloader_pin_memory       = True,
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
    label_names                 = ["labels"],
)

assert training_args.fp16 == True, "FATAL: fp16 must be True for T4."
logger.info("✅ training_args.fp16 assertion passed.")
# Confirm output_dir is still correct after TrainingArguments construction
assert training_args.output_dir == CHECKPOINT_DIR, (
    f"output_dir mismatch: {training_args.output_dir} != {CHECKPOINT_DIR}"
)
logger.info(
    f"TrainingArguments configured. output_dir={training_args.output_dir} | "
    f"save_strategy=steps | save_steps=50 | save_total_limit=3"
)

# Training

In [ ]:
# ==========================
# Block 5: Pre-Tokenization, Root Cleanup, Checkpoint Resume & Training Loop
# Version: v1.4
# Purpose: Clean stray root-level checkpoint dirs before training, pre-tokenize
#          both datasets, detect and resume from latest checkpoint, instantiate
#          DEITrainer with all components, run the training loop, then delete
#          step checkpoints leaving only best_model/.
# ==========================
#
# What this block does: (~440 words)
#
# - Feature 1 (Root Cleanup ✅ — NEW in v1.4): cleanup_root_debris() runs
#   BEFORE training starts. It scans /content/drive/MyDrive directly for any
#   directory whose name starts with "checkpoint-" or matches a known model
#   folder name (qwen3.5_4b, qwen3.5_9b, qwen3.5_4b_merged). These are
#   artefacts from prior sessions where CHECKPOINT_DIR was wrong. Critically,
#   it EXCLUDES anything under _FINETUNE_ROOT (those are legitimate saves).
#   It uses shutil.rmtree with ignore_errors=True and logs every deletion.
#   This cleanup runs BEFORE instantiating the trainer so that find_latest_
#   checkpoint() only sees legitimate checkpoints inside CHECKPOINT_DIR.
#
# - Feature 2 (Pre-Tokenization ✅): Unchanged from v1.3. trl 0.13.0 removed
#   internal tokenization. Explicit .map() over both HF datasets with
#   truncation=True, max_length=768, padding=False. Tweet chars capped at 800
#   to prevent response template truncation.
#
# - Feature 3 (Checkpoint Detection ✅): find_latest_checkpoint() scans
#   CHECKPOINT_DIR for directories matching checkpoint-<int>, skips .tmp/.old,
#   verifies adapter_config.json exists. Returns highest-step valid path.
#
# - Feature 4 (Custom State Restore ✅): If resuming, loads custom_state.pt
#   to restore best_f1 into trainer and callback.
#
# - Feature 5 (Security Patch ✅): Patches check_torch_load_is_safe() in
#   transformers.trainer to avoid torch.load security exception on PyTorch
#   2.5.1.
#
# - Feature 6 (Training ✅): trainer.train(resume_from_checkpoint=path) if
#   checkpoint found, else trainer.train() from scratch. The new save_model()
#   override in DEITrainer ensures no unauthorized saves occur during the loop.
#
# - Feature 7 (Post-Training Cleanup ✅): Unchanged from v1.3. Deletes all
#   checkpoint-N step directories after training completes, leaving only
#   best_model/ inside CHECKPOINT_DIR.
#
# How it works:
# - Step 1: Run cleanup_root_debris() to remove stale root-level dirs.
# - Step 2: Set PYTORCH_CUDA_ALLOC_CONF, flush GPU memory.
# - Step 3: Pre-tokenize train + val datasets.
# - Step 4: find_latest_checkpoint(), instantiate trainer + callback.
# - Step 5: Restore custom state if resuming.
# - Step 6: Apply security patch, final memory flush.
# - Step 7: trainer.train().
# - Step 8: Delete step checkpoints, log survivors.
#
# Solutions attempted that did not work:
# - v1.3: No root cleanup — stale checkpoint-N dirs from prior sessions
#   remained at Drive root, wasting quota and confusing find_latest_checkpoint
#   if it ever scanned the wrong directory. Fixed in v1.4 by pre-training cleanup.
# - v1.3: No save_model override (in DEITrainer) — end-of-training Trainer call
#   to save_model(output_dir) could write to unauthorized locations. Fixed in
#   Block 4 v1.3.
#
# Solutions implemented:
# - cleanup_root_debris() before training.
# - All saves validated by Block 1 path guard + Block 4 save_model override.
# ==========================

print("====++++++====")
print("Block 5 Output")
print("====++++++====")
print()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, logging, shutil, torch
import transformers.trainer, transformers.utils.import_utils

logger = logging.getLogger("dei_finetune")

# ── ROOT DEBRIS CLEANUP (NEW v1.4) ─────────────────────────────────────────
_DEBRIS_PATTERNS = ("checkpoint-", "qwen3.5_4b", "qwen3.5_9b", "qwen3.5_4b_merged")

def cleanup_root_debris(
    drive_root: str      = "/content/drive/MyDrive",
    finetune_root: str   = _FINETUNE_ROOT,
    dry_run: bool        = False,
) -> None:
    """
    Deletes stray checkpoint / model directories sitting at Drive root level.
    Explicitly SKIPS anything whose resolved path starts with finetune_root
    (those are legitimate saves). Logs every action. Use dry_run=True to
    preview without deleting.
    """
    logger.info(
        f"{'[DRY RUN] ' if dry_run else ''}Scanning for root debris in: {drive_root}"
    )
    removed, skipped = 0, 0
    try:
        for name in sorted(os.listdir(drive_root)):
            path = os.path.join(drive_root, name)
            if not os.path.isdir(path):
                continue
            # Only target known debris patterns
            is_target = any(name.startswith(p) or name == p for p in _DEBRIS_PATTERNS)
            if not is_target:
                continue
            # Skip anything inside (or equal to) the legitimate finetune root
            abs_path = os.path.abspath(path)
            if abs_path.startswith(os.path.abspath(finetune_root)):
                logger.info(f"  Skipping (inside finetune root): {path}")
                skipped += 1
                continue
            if dry_run:
                logger.info(f"  [DRY RUN] Would remove: {path}")
            else:
                shutil.rmtree(path, ignore_errors=True)
                logger.info(f"  ✅ Removed root debris: {path}")
            removed += 1
    except Exception as _e:
        logger.error(f"cleanup_root_debris error: {_e}")

    logger.info(
        f"Root debris cleanup complete — "
        f"{'would remove' if dry_run else 'removed'}: {removed} dir(s) | "
        f"skipped (legitimate): {skipped} dir(s)."
    )

# Run cleanup BEFORE any training object is instantiated.
# Set dry_run=True first to preview, then False to actually delete.
cleanup_root_debris(dry_run=False)

# ── MEMORY CLEAR ───────────────────────────────────────────────────────────
logger.info("Clearing GPU memory...")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
logger.info(
    f"VRAM after clear — "
    f"allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB | "
    f"reserved:  {torch.cuda.memory_reserved(0)  / 1024**3:.2f} GB"
)

training_args.per_device_eval_batch_size = 4
logger.info("per_device_eval_batch_size overridden to 4 (OOM prevention).")

# ── PRE-TOKENIZE DATASETS ──────────────────────────────────────────────────
logger.info("Pre-tokenizing train and val datasets (truncation=True, max_length=768)...")
_MAX_TWEET_CHARS = 800

def _tokenize_batch(examples: dict) -> dict:
    capped = []
    for text in examples["text"]:
        tweet_start = text.find("Tweet: ")
        tweet_end   = text.find("<|im_end|>", tweet_start) if tweet_start != -1 else -1
        if tweet_start != -1 and tweet_end != -1:
            tweet_content = text[tweet_start + 7 : tweet_end]
            if len(tweet_content) > _MAX_TWEET_CHARS:
                text = (
                    text[:tweet_start + 7]
                    + tweet_content[:_MAX_TWEET_CHARS]
                    + text[tweet_end:]
                )
        capped.append(text)
    return tokenizer(capped, truncation=True, max_length=768, padding=False)

train_tokenized = train_hf_dataset.map(
    _tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing train",
)
val_tokenized = val_hf_dataset.map(
    _tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing val",
)
logger.info(
    f"Tokenization complete — train: {len(train_tokenized)} | "
    f"val: {len(val_tokenized)} | columns: {train_tokenized.column_names}"
)

# ── CHECKPOINT DETECTION ───────────────────────────────────────────────────
def find_latest_checkpoint(checkpoint_dir: str) -> str | None:
    if not os.path.exists(checkpoint_dir):
        return None
    candidates = []
    for name in os.listdir(checkpoint_dir):
        if not name.startswith("checkpoint-"):
            continue
        if name.endswith(".tmp") or name.endswith(".old"):
            continue
        step_str = name.split("-")[-1]
        if not step_str.isdigit():
            continue
        ckpt_path = os.path.join(checkpoint_dir, name)
        if not os.path.isfile(os.path.join(ckpt_path, "adapter_config.json")):
            logger.warning(f"Skipping corrupt checkpoint (no adapter_config.json): {ckpt_path}")
            continue
        candidates.append((int(step_str), ckpt_path))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    latest_step, latest_path = candidates[-1]
    logger.info(f"Latest checkpoint: {latest_path} (step {latest_step})")
    return latest_path

checkpoint_path = find_latest_checkpoint(CHECKPOINT_DIR)

# ── INSTANTIATE TRAINER ────────────────────────────────────────────────────
monitor_callback = TrainingMonitorCallback(
    best_model_dir=BEST_MODEL_DIR, patience=2, threshold=0.001,
)
training_args.dataloader_num_workers = 0

trainer = DEITrainer(
    model                         = model,
    args                          = training_args,
    train_dataset                 = train_tokenized,
    eval_dataset                  = val_tokenized,
    processing_class              = tokenizer,
    data_collator                 = collator,
    class_weights                 = class_weights_tensor,
    label_token_ids_map           = label_token_ids,
    compute_metrics               = compute_metrics_fn,
    preprocess_logits_for_metrics = preprocess_logits_for_metrics,
    callbacks                     = [monitor_callback],
)
trainer._monitor_callback_ref = monitor_callback

# ── RESTORE CUSTOM STATE ───────────────────────────────────────────────────
if checkpoint_path is not None:
    _custom_state_path = os.path.join(checkpoint_path, "custom_state.pt")
    if os.path.isfile(_custom_state_path):
        _cs = torch.load(_custom_state_path, map_location="cpu", weights_only=True)
        trainer.best_f1          = _cs.get("best_f1", 0.0)
        monitor_callback.best_f1 = _cs.get("best_f1", 0.0)
        logger.info(
            f"Custom state restored — best_f1: {trainer.best_f1:.4f} | "
            f"epoch: {_cs.get('epoch','?')} | step: {_cs.get('global_step','?')}"
        )
    else:
        logger.warning("custom_state.pt not found — best_f1 recovered from log_history.")
    logger.info(f"Resuming from: {checkpoint_path}")
else:
    logger.info("No checkpoint found — starting fresh from step 0.")

# ── SECURITY PATCH ─────────────────────────────────────────────────────────
if hasattr(transformers.trainer, "check_torch_load_is_safe"):
    transformers.trainer.check_torch_load_is_safe = lambda: None
if hasattr(transformers.utils.import_utils, "is_torch_greater_or_equal"):
    _orig = transformers.utils.import_utils.is_torch_greater_or_equal
    transformers.utils.import_utils.is_torch_greater_or_equal = (
        lambda v: True if v == "2.6" else _orig(v)
    )

# ── FINAL MEMORY FLUSH ─────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
logger.info(
    f"VRAM before train() — "
    f"allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB | "
    f"reserved:  {torch.cuda.memory_reserved(0)  / 1024**3:.2f} GB"
)

# ── RUN TRAINING ───────────────────────────────────────────────────────────
logger.info(
    f"Starting training — epochs: {training_args.num_train_epochs} | "
    f"effective batch: "
    f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} | "
    f"steps per epoch: "
    f"{len(train_tokenized) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}"
)

if checkpoint_path is not None:
    train_result = trainer.train(resume_from_checkpoint=checkpoint_path)
else:
    train_result = trainer.train()

logger.info(f"Training complete. Metrics: {train_result.metrics}")
logger.info(f"Final best F1: {monitor_callback.best_f1:.4f}")

# ── POST-TRAINING CLEANUP — delete step checkpoints, keep best_model/ ──────
logger.info("Post-training cleanup: removing step checkpoints...")
_best_model_realpath = os.path.realpath(BEST_MODEL_DIR)
_removed_count, _skipped_count = 0, 0
for _name in sorted(os.listdir(CHECKPOINT_DIR)):
    _path = os.path.join(CHECKPOINT_DIR, _name)
    if not os.path.isdir(_path):
        _skipped_count += 1
        continue
    if not (_name.startswith("checkpoint-") and _name.split("-")[-1].isdigit()):
        logger.info(f"  Skipped (not step checkpoint): {_name}/")
        _skipped_count += 1
        continue
    if os.path.realpath(_path) == _best_model_realpath:
        logger.warning(f"  Skipped (resolves to best_model/): {_name}/")
        _skipped_count += 1
        continue
    shutil.rmtree(_path, ignore_errors=True)
    logger.info(f"  Removed: {_name}/")
    _removed_count += 1

logger.info(
    f"Cleanup complete — removed {_removed_count} step checkpoint(s), "
    f"skipped {_skipped_count} item(s)."
)
_survivors = sorted(os.listdir(CHECKPOINT_DIR))
logger.info(f"CHECKPOINT_DIR contains {len(_survivors)} item(s):")
for _s in _survivors:
    _sp   = os.path.join(CHECKPOINT_DIR, _s)
    _kind = "dir" if os.path.isdir(_sp) else "file"
    logger.info(f"  [{_kind}] {_s}")
logger.info(f"✅ Block 5 v1.4 Complete. Best model at: {BEST_MODEL_DIR}")

In [ ]:
# ==========================
# Block 6: Post-Training Generative Evaluation
# Version: v3.0
# Purpose: Load best adapter, run full generative evaluation on val set using
#          greedy decoding, compute F1/precision/recall/AUC-ROC, log confusion
#          matrix, run threshold sensitivity, and save results CSV + JSON.
# ==========================
#
# What this block does: (~420 words)
#
# - Feature 1 (Adapter Load Guard ✅): Checks if best_model/ directory contains
#   adapter_config.json. If yes, calls load_adapter() wrapped in try/except to
#   handle AdapterAlreadyLoadedException from PEFT 0.15.0 which prohibits loading
#   over an existing active adapter. The except branch logs a warning and proceeds
#   with the current model weights — since TrainingMonitorCallback saves best_model/
#   by calling model.save_pretrained(), the current in-memory weights ARE the best
#   adapter after training completes in the same session.
#
# - Feature 2 (Left Padding for Batched Generation ✅): Switches padding_side to
#   "left" before generation and restores to "right" in a finally block. Decoder-only
#   models require left-padding for correct position IDs during batched autoregressive
#   generation. Right-padding causes position ID misalignment and produces silent
#   garbage in batch positions beyond the first.
#
# - Feature 3 (Greedy Decoding ✅ — v2.0→v3.0 fix): do_sample=False. For binary
#   classification with two valid output tokens, sampling introduces noise with zero
#   benefit. Greedy picks the highest-probability token at each step, which for a
#   fine-tuned classifier is always the correct strategy. Sampling was the primary
#   cause of parse failures >2% in v2.0.
#
# - Feature 4 (max_new_tokens=3 ✅): Label is 1 token. EOS is 1 token. 1 margin
#   token. Total 3 is sufficient and prevents the model from generating explanations
#   or thinking tokens before EOS, which inflate parse failures.
#
# - Feature 5 (AUC-ROC from first-token logits ✅): Extracts raw logit scores for
#   token_id_0 and token_id_1 at the first generated position, computes softmax to
#   get P(label=1), and passes to roc_auc_score. This gives a proper probability-based
#   ranking metric independent of the hard classification threshold.
#
# - Feature 6 (Parse Failure Logging ✅): Any output where the first meaningful
#   character after strip_think_tokens() is not "0" or "1" is a parse failure.
#   These are excluded from metrics but included in results CSV with
#   parse_failure=True. If failure rate >2%, a warning explains the root cause.
#
# - Feature 7 (Threshold Sensitivity ✅): After evaluation, sweeps classification
#   thresholds from 0.50 to 0.80 using the already-computed AUC scores, printing
#   F1, precision, recall, FP, FN at each threshold. No re-inference needed.
#   Recommended threshold 0.65 to compensate for training prevalence (varies by
#   dataset) vs real-world prevalence mismatch.
#
# - Feature 8 (Results Persistence ✅): Saves val_predictions.csv and
#   eval_metrics.json to RESULTS_DIR. eval_metrics.json includes generation
#   parameters for reproducibility.
#
# How it works:
# - Step 1: Load best adapter or fall back to current weights.
# - Step 2: Switch padding_side to "left".
# - Step 3: For each batch: tokenize prompts, generate greedy with max_new_tokens=3,
#           extract first-token logits for AUC-ROC, decode and parse output.
# - Step 4: Restore padding_side to "right" in finally block.
# - Step 5: Compute all metrics on valid predictions.
# - Step 6: Log confusion matrix, save CSV + JSON.
# - Step 7: Run threshold sensitivity sweep.
#
# Solutions attempted that did not work:
# - v2.0: do_sample=True, temperature=0.7 — inflated parse failures for binary task.
# - v2.0: load_adapter() without try/except — AdapterAlreadyLoadedException crash.
#
# Solutions implemented:
# - v3.0: Greedy decoding. try/except around adapter load.

print("====++++++====")
print("Block 6 Output")
print("====++++++====")
print()

import json
import logging
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
)
from tqdm import tqdm

logger = logging.getLogger("dei_finetune")
EVAL_BATCH_SIZE = 16

# ── parse_model_output ─────────────────────────────────────────────────────
def parse_model_output(raw_output: str) -> int:
    stripped = strip_think_tokens(raw_output).strip()
    if not stripped:
        return -1
    if stripped[0] == "0":
        return 0
    if stripped[0] == "1":
        return 1
    return -1

# ── Load best adapter safely ───────────────────────────────────────────────
_best_adapter_config = os.path.join(BEST_MODEL_DIR, "adapter_config.json")
_eval_model = model

if os.path.isfile(_best_adapter_config):
    logger.info(f"Best adapter found at {BEST_MODEL_DIR}. Attempting to load...")
    try:
        _eval_model.load_adapter(BEST_MODEL_DIR, adapter_name="best")
        _eval_model.set_adapter("best")
        logger.info("✅ Best adapter loaded and set as active.")
    except Exception as _adapter_err:
        logger.warning(
            f"load_adapter() failed: {_adapter_err}. "
            "Evaluating with current in-memory weights "
            "(these ARE the best adapter — TrainingMonitorCallback saved them)."
        )
else:
    logger.warning(
        f"best_model/ not found at {BEST_MODEL_DIR}. "
        "Evaluating with final checkpoint model weights."
    )

# ── run_generative_eval ────────────────────────────────────────────────────
def run_generative_eval(
    eval_model,
    eval_tokenizer,
    eval_df: pd.DataFrame,
    batch_size: int = EVAL_BATCH_SIZE,
):
    original_padding_side = eval_tokenizer.padding_side
    eval_tokenizer.padding_side = "left"
    logger.info(
        "Switched tokenizer.padding_side to 'left' for batched generation."
    )
    eval_model.eval()
    prompts     = [format_inference_prompt(t) for t in eval_df["Tweet_clean"].tolist()]
    true_labels = eval_df["DEI"].tolist()
    predictions:    list[int]   = []
    scores:         list[float] = []
    raw_outputs:    list[str]   = []
    parse_failures: list[dict]  = []

    try:
        for batch_start in tqdm(
            range(0, len(prompts), batch_size),
            desc="Generative eval",
            total=(len(prompts) + batch_size - 1) // batch_size,
        ):
            batch_prompts = prompts[batch_start : batch_start + batch_size]
            batch_labels  = true_labels[batch_start : batch_start + batch_size]

            enc = eval_tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=768,
            ).to(eval_model.device)

            with torch.no_grad():
                gen_out = eval_model.generate(
                    **enc,
                    max_new_tokens=3,
                    do_sample=False,
                    pad_token_id=eval_tokenizer.eos_token_id,
                    eos_token_id=eval_tokenizer.eos_token_id,
                    return_dict_in_generate=True,
                    output_scores=True,
                )

            input_len    = enc["input_ids"].shape[1]
            _first_logits = gen_out.scores[0]
            _token_ids   = torch.tensor(
                [label_token_ids[0], label_token_ids[1]],
                device=_first_logits.device,
            )
            _pair_logits = _first_logits[:, _token_ids]
            _probs       = torch.softmax(_pair_logits, dim=-1)
            _p1          = _probs[:, 1].cpu().float().tolist()

            for j, (gen_seq, true_lbl, p1) in enumerate(
                zip(gen_out.sequences, batch_labels, _p1)
            ):
                gen_tokens = gen_seq[input_len:]
                raw_text   = eval_tokenizer.decode(
                    gen_tokens, skip_special_tokens=True
                )
                raw_outputs.append(raw_text)
                scores.append(float(p1))
                parsed = parse_model_output(raw_text)
                predictions.append(parsed)
                if parsed == -1:
                    parse_failures.append({
                        "global_index": batch_start + j,
                        "raw_output":   raw_text,
                        "true_label":   true_lbl,
                    })
    finally:
        eval_tokenizer.padding_side = original_padding_side
        logger.info(
            f"Restored tokenizer.padding_side to '{original_padding_side}'."
        )
        eval_model.train()

    return predictions, true_labels, scores, raw_outputs, parse_failures

# ── Run evaluation ─────────────────────────────────────────────────────────
logger.info("Running post-training generative evaluation on validation set...")
preds, trues, auc_scores, raw_outs, failures = run_generative_eval(
    eval_model=_eval_model,
    eval_tokenizer=tokenizer,
    eval_df=val_df,
)

# ── Parse failure analysis ─────────────────────────────────────────────────
n_total_eval = len(preds)
n_failures   = len(failures)
failure_rate = n_failures / n_total_eval * 100
logger.info(
    f"Parse failure rate: {n_failures}/{n_total_eval} = {failure_rate:.2f}%"
)
if failure_rate > 2.0:
    logger.warning(
        f"Parse failure rate {failure_rate:.2f}% exceeds 2% threshold. "
        "Inspect raw_outputs — if they start with '<think>', SFT did not fully "
        "converge on binary output format. Check training loss curve."
    )
    for _i, _fail in enumerate(failures[:5]):
        logger.warning(
            f"Failure {_i+1}: idx={_fail['global_index']} | "
            f"true={_fail['true_label']} | raw={_fail['raw_output']!r}"
        )
else:
    logger.info(f"✅ Parse failure rate {failure_rate:.2f}% ≤ 2%.")

# ── Compute metrics ────────────────────────────────────────────────────────
valid_mask_eval = [p != -1 for p in preds]
preds_valid     = [p for p, m in zip(preds, valid_mask_eval) if m]
trues_valid     = [t for t, m in zip(trues, valid_mask_eval) if m]
scores_valid    = [s for s, m in zip(auc_scores, valid_mask_eval) if m]

eval_metrics = {}
if len(preds_valid) == 0:
    logger.error("All predictions are parse failures — no metrics to compute.")
else:
    acc      = accuracy_score(trues_valid, preds_valid)
    f1_macro = f1_score(trues_valid, preds_valid, average="macro",    zero_division=0)
    f1_cls0  = f1_score(trues_valid, preds_valid, average=None, zero_division=0, labels=[0])[0]
    f1_cls1  = f1_score(trues_valid, preds_valid, average=None, zero_division=0, labels=[1])[0]
    prec0    = precision_score(trues_valid, preds_valid, pos_label=0, zero_division=0)
    prec1    = precision_score(trues_valid, preds_valid, pos_label=1, zero_division=0)
    rec0     = recall_score(trues_valid, preds_valid, pos_label=0, zero_division=0)
    rec1     = recall_score(trues_valid, preds_valid, pos_label=1, zero_division=0)
    try:
        auc_roc = roc_auc_score(trues_valid, scores_valid)
    except ValueError as _e:
        auc_roc = float("nan")
        logger.warning(f"AUC-ROC could not be computed: {_e}")
    cm = confusion_matrix(trues_valid, preds_valid, labels=[0, 1])

    logger.info("=" * 60)
    logger.info(
        "POST-TRAINING GENERATIVE EVALUATION RESULTS (v3.0 — greedy decoding)"
    )
    logger.info("=" * 60)
    logger.info(
        f"Evaluated on {len(preds_valid)}/{n_total_eval} "
        f"({failure_rate:.2f}% parse failures excluded)"
    )
    logger.info(f"Accuracy:              {acc:.4f}")
    logger.info(f"F1 macro:              {f1_macro:.4f}")
    logger.info(f"F1 class 0 (DEI-neg):  {f1_cls0:.4f}")
    logger.info(f"F1 class 1 (DEI-pos):  {f1_cls1:.4f}")
    logger.info(f"Precision class 0:     {prec0:.4f}")
    logger.info(f"Precision class 1:     {prec1:.4f}")
    logger.info(f"Recall class 0:        {rec0:.4f}")
    logger.info(f"Recall class 1:        {rec1:.4f}")
    logger.info(f"AUC-ROC:               {auc_roc:.4f}")
    logger.info(f"Parse failure rate:    {failure_rate:.2f}%")
    logger.info(
        f"Confusion Matrix (rows=true, cols=pred):\n"
        f"           Pred=0  Pred=1\n"
        f"True=0     {cm[0,0]:5d}   {cm[0,1]:5d}\n"
        f"True=1     {cm[1,0]:5d}   {cm[1,1]:5d}"
    )
    logger.info("=" * 60)

    if rec1 >= 0.50 and f1_macro >= 0.80:
        logger.info(
            f"✅ Recall class 1: {rec1:.4f} | F1 macro: {f1_macro:.4f} — "
            "Model is ready for inference."
        )
    elif rec1 < 0.50:
        logger.warning(
            f"DEI-positive Recall {rec1:.4f} < 0.50. "
            "Consider adding more DEI-positive examples or increasing epochs."
        )

    # ── Save results ───────────────────────────────────────────────────────
    results_df = val_df[["Tweet_clean", "DEI"]].copy()
    results_df["prediction"]    = preds
    results_df["auc_score"]     = auc_scores
    results_df["raw_output"]    = raw_outs
    results_df["parse_failure"] = [p == -1 for p in preds]
    results_df.to_csv(
        os.path.join(RESULTS_DIR, "val_predictions.csv"), index=False
    )

    eval_metrics = {
        "accuracy":               round(acc, 6),
        "f1_macro":               round(f1_macro, 6),
        "f1_class_0":             round(float(f1_cls0), 6),
        "f1_class_1":             round(float(f1_cls1), 6),
        "precision_class_0":      round(float(prec0), 6),
        "precision_class_1":      round(float(prec1), 6),
        "recall_class_0":         round(float(rec0), 6),
        "recall_class_1":         round(float(rec1), 6),
        "auc_roc":                round(float(auc_roc), 6) if not np.isnan(auc_roc) else None,
        "parse_failure_rate_pct": round(failure_rate, 4),
        "n_valid":                len(preds_valid),
        "n_failures":             n_failures,
        "n_total":                n_total_eval,
        "val_size":               len(val_df),
        "train_size":             len(train_df),
        "val_split_ratio":        0.20,
        "confusion_matrix":       cm.tolist(),
        "generation_params": {
            "max_new_tokens": 3,
            "do_sample":      False,
            "decoding":       "greedy",
        },
    }
    with open(os.path.join(RESULTS_DIR, "eval_metrics.json"), "w") as _f:
        json.dump(eval_metrics, _f, indent=2)
    logger.info(
        f"Results saved: {RESULTS_DIR}/val_predictions.csv | "
        f"{RESULTS_DIR}/eval_metrics.json"
    )

logger.info("✅ Block 6 v3.0 Complete.")

# ── Threshold Sensitivity ──────────────────────────────────────────────────
print()
print("=" * 65)
print("THRESHOLD SENSITIVITY — Val Set")
print("=" * 65)
print(
    f"{'Threshold':>10} | {'F1':>6} | {'Prec1':>6} | "
    f"{'Rec1':>6} | {'FP':>4} | {'FN':>4}"
)
print("-" * 65)
for thresh in [0.50, 0.55, 0.60, 0.62, 0.65, 0.68, 0.70, 0.72, 0.75, 0.78, 0.80]:
    preds_t = [1 if s >= thresh else 0 for s in auc_scores]
    f1   = f1_score(trues, preds_t, average="macro",  zero_division=0)
    p1   = precision_score(trues, preds_t, pos_label=1, zero_division=0)
    r1   = recall_score(trues, preds_t, pos_label=1, zero_division=0)
    cm_t = confusion_matrix(trues, preds_t, labels=[0, 1])
    fp   = cm_t[0, 1]
    fn   = cm_t[1, 0]
    flag = " ← current" if thresh == 0.65 else ""
    print(
        f"{thresh:>10.2f} | {f1:>6.4f} | {p1:>6.4f} | "
        f"{r1:>6.4f} | {fp:>4} | {fn:>4}{flag}"
    )
print("=" * 65)
print("For corpus completeness : prioritise Recall (low FN).")
print("For labelling efficiency: prioritise Precision (low FP).")

In [ ]:
# ── THRESHOLD SENSITIVITY ANALYSIS ────────────────────────────────────────
# Run this AFTER Block 6 completes. Uses the scores already computed.
# No retraining, no re-inference needed.

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

print("=" * 65)
print("THRESHOLD SENSITIVITY — Val Set (400 rows)")
print("=" * 65)
print(f"{'Threshold':>10} | {'F1':>6} | {'Prec1':>6} | {'Rec1':>6} | {'FP':>4} | {'FN':>4}")
print("-" * 65)

# `auc_scores` and `trues` are already in memory from Block 6
for thresh in [0.50, 0.55, 0.60, 0.62, 0.65, 0.68, 0.70, 0.72, 0.75, 0.78, 0.80]:
    preds_t = [1 if s >= thresh else 0 for s in auc_scores]
    f1   = f1_score(trues, preds_t, average="macro", zero_division=0)
    p1   = precision_score(trues, preds_t, pos_label=1, zero_division=0)
    r1   = recall_score(trues, preds_t, pos_label=1, zero_division=0)
    cm   = confusion_matrix(trues, preds_t, labels=[0, 1])
    fp   = cm[0, 1]
    fn   = cm[1, 0]
    flag = " ← current" if thresh == 0.65 else ""
    print(f"{thresh:>10.2f} | {f1:>6.4f} | {p1:>6.4f} | {r1:>6.4f} | {fp:>4} | {fn:>4}{flag}")

print("=" * 65)
print("Pick the threshold where FP drops without FN spiking badly.")
print("For corpus completeness: prioritise Recall (low FN).")
print("For Gemini labelling efficiency: prioritise Precision (low FP).")

In [ ]:
# ==========================
# Block 7: Save & Export LoRA Adapter to Drive
# Version: v1.1
# Purpose: Save the final LoRA adapter weights, tokenizer, and a full run_config.json
#          documenting all hyperparameters, split config, results, and inference spec.
# ==========================
#
# What this block does: (~350 words)
#
# - Feature 1 (Adapter-Only Save ✅): Calls model.save_pretrained(ADAPTER_DIR) on
#   a PeftModel, which saves only the LoRA delta — adapter_config.json and
#   adapter_model.safetensors — NOT the 4B base model weights. The 4B adapter is
#   ~55 MB; the full model is ~8 GB. Saving only the adapter is intentional: Phase 2
#   inference loads the base model separately and merges at runtime, giving maximum
#   precision flexibility (bf16 on better GPUs, fp16 on T4 equivalents).
#
# - Feature 2 (Tokenizer Save ✅): Saves tokenizer alongside adapter including the
#   custom minimal chat_template.jinja. This ensures the inference pipeline uses
#   the exact same template the fine-tuned model was trained with, not the original
#   7k-character thinking-enabled template from HuggingFace.
#
# - Feature 3 (run_config.json ✅): Writes a comprehensive JSON record of the
#   entire training run: base model name, LoRA config, quantization settings,
#   data config (single file, split ratio, train/val sizes, class weights),
#   training hyperparameters (lr, scheduler, batch, epochs, seed), and results
#   (final F1, parse failure rate, confusion matrix). This serves as the
#   ground truth for reproducing the exact training run from scratch.
#   v1.1 adds split_config section documenting that val was created in-memory
#   from merged_4k_v2_train.csv with stratified 80/20 split.
#
# - Feature 4 (Directory Verification ✅): Lists all files in ADAPTER_DIR with
#   sizes after save. The expected contents are: README.md, adapter_config.json,
#   adapter_model.safetensors (~55 MB), chat_template.jinja, run_config.json,
#   tokenizer.json (~19 MB), tokenizer_config.json.
#
# How it works:
# - Step 1: model.save_pretrained(ADAPTER_DIR).
# - Step 2: tokenizer.save_pretrained(ADAPTER_DIR).
# - Step 3: Build and write run_config.json.
# - Step 4: List directory contents with file sizes.
#
# Solutions attempted that did not work:
# - Saving merged model (merge_and_unload() then save_pretrained) — produces ~16 GB
#   file, saturates Drive quota, and locks in quantization level. Rejected.
#
# Solutions implemented:
# - v1.1: Adapter-only save. run_config.json includes split_config section
#   documenting the in-memory stratified split, absent in v1.0.

print("====++++++====")
print("Block 7 Output")
print("====++++++====")
print()

import json
import logging
import os
from datetime import datetime, timezone
from tqdm import tqdm

logger = logging.getLogger("dei_finetune")
os.makedirs(ADAPTER_DIR, exist_ok=True)

# ── Save LoRA Adapter ──────────────────────────────────────────────────────
logger.info(f"Saving LoRA adapter to: {ADAPTER_DIR}")
for _step in tqdm(["save_adapter", "save_tokenizer"], desc="Exporting"):
    if _step == "save_adapter":
        model.save_pretrained(ADAPTER_DIR)
    else:
        tokenizer.save_pretrained(ADAPTER_DIR)
logger.info("Adapter and tokenizer saved.")

# ── Build run_config.json ──────────────────────────────────────────────────
_final_f1   = getattr(monitor_callback, "best_f1", 0.0) or trainer.best_f1
_parse_rate = eval_metrics.get("parse_failure_rate_pct", None) if eval_metrics else None

_best_epoch = None
for _entry in reversed(trainer.state.log_history):
    if abs(_entry.get("eval_f1_macro", -1) - _final_f1) < 1e-6:
        _best_epoch = _entry.get("epoch")
        break

run_config = {
    "project":       "DEI_tweet_classification",
    "version":       "v1.1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "base_model":    MODEL_NAME,
    "adapter_path":  ADAPTER_DIR,
    "split_config": {
        "source_file":        RAW_DATA_PATH,
        "strategy":           "stratified_in_memory",
        "val_split_ratio":    0.20,
        "train_size":         len(train_df),
        "val_size":           len(val_df),
        "seed":               SEED,
        "note": (
            "No separate val file. Val created in Block 1.5 via "
            "sklearn train_test_split(stratify=DEI, random_state=42)."
        ),
    },
    "training": {
        "num_train_epochs":            training_args.num_train_epochs,
        "per_device_train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "effective_batch_size": (
            training_args.per_device_train_batch_size
            * training_args.gradient_accumulation_steps
        ),
        "learning_rate":          training_args.learning_rate,
        "weight_decay":           training_args.weight_decay,
        "lr_scheduler_type":      training_args.lr_scheduler_type,
        "warmup_ratio":           training_args.warmup_ratio,
        "max_seq_length":         768,
        "fp16":                   training_args.fp16,
        "bf16":                   training_args.bf16,
        "optim":                  training_args.optim,
        "gradient_checkpointing": training_args.gradient_checkpointing,
        "seed":                   SEED,
    },
    "lora": {
        "r":              lora_config.r,
        "lora_alpha":     lora_config.lora_alpha,
        "lora_dropout":   lora_config.lora_dropout,
        "bias":           lora_config.bias,
        "target_modules": list(lora_config.target_modules),
    },
    "quantisation": {
        "load_in_4bit":              True,
        "bnb_4bit_quant_type":       "nf4",
        "bnb_4bit_use_double_quant": True,
        "bnb_4bit_compute_dtype":    "float16",
    },
    "data": {
        "class_weight_0":     float(class_weights_tensor[0]),
        "class_weight_1":     float(class_weights_tensor[1]),
        "label_token_id_0":   label_token_ids[0],
        "label_token_id_1":   label_token_ids[1],
    },
    "results": {
        "final_f1_macro":         round(float(_final_f1), 6) if _final_f1 else None,
        "best_epoch":             _best_epoch,
        "parse_failure_rate_pct": _parse_rate,
        "eval_metrics":           eval_metrics if eval_metrics else {},
    },
    "inference": {
        "parser_function": "parse_model_output",
        "parser_spec":     "strip_think_tokens → first char → 0/1/-1",
        "threshold":       0.50,
        "phase2_note": (
            "Load Qwen/Qwen3.5-4B + this adapter via PeftModel. "
            "Use greedy decoding, max_new_tokens=3. "
            "Apply parse_model_output() to each output."
        ),
    },
}

_config_path = os.path.join(ADAPTER_DIR, "run_config.json")
with open(_config_path, "w") as _f:
    json.dump(run_config, _f, indent=2)
logger.info(f"run_config.json saved to: {_config_path}")

# ── Verify Output ──────────────────────────────────────────────────────────
_files = sorted(os.listdir(ADAPTER_DIR))
logger.info(f"Adapter directory contents ({len(_files)} files):")
for _fname in tqdm(_files, desc="Listing adapter dir"):
    _fpath = os.path.join(ADAPTER_DIR, _fname)
    _size  = os.path.getsize(_fpath) / (1024 ** 2)
    logger.info(f"  {_fname:<40s}  {_size:.2f} MB")

logger.info("✅ Block 7 v1.1 Complete. Adapter ready for Phase 2 inference.")

In [ ]:
# ==========================
# Block 8: (Optional) Cross-Validation Runner
# Version: v1.0
# Purpose: Run 5-fold stratified cross-validation using in-memory splits of the
#          single merged_4k_v2_train.csv file; report mean ± std F1 macro.
# ==========================
#
# What this block does: (~380 words)
#
# - Feature 1 (USE_CV Guard ✅): CV is disabled by default (USE_CV=False). It is
#   only warranted if 3+ independent main-pipeline runs with different seeds produce
#   F1 std >0.03 on the single stratified split. At ~4000 training examples, a
#   single 80/20 stratified split produces a val set of ~1000 examples — large enough
#   for reliable F1 estimation. CV costs 5× the compute for marginal additional
#   insight at this dataset size. Default: use single split. CV is the escalation path.
#
# - Feature 2 (Full Dataset for CV ✅): CV folds are created from full_df (the
#   complete dataset before the 80/20 split), NOT from train_df. Using train_df would
#   exclude the 20% val examples from CV entirely, producing biased fold estimates.
#   StratifiedKFold on full_df gives each fold access to all available examples.
#
# - Feature 3 (Fresh Model Per Fold ✅): Each fold reloads the base model from
#   scratch via AutoModelForCausalLM.from_pretrained() with BnB config. This is
#   necessary to prevent weight leakage: if fold N adapter weights remain in memory
#   when fold N+1 starts, the new LoRA layers are initialized on top of already-
#   adapted weights rather than the base model, making fold N+1 results dependent on
#   fold N training history. GPU memory is explicitly freed with gc.collect() and
#   torch.cuda.empty_cache() between folds.
#
# - Feature 4 (3-Epoch CV ✅): CV folds train for 3 epochs vs 5 in the main pipeline.
#   The goal of CV is variance estimation, not model selection — 3 epochs is sufficient
#   to measure whether the model converges similarly across folds. Training all folds
#   to epoch 5 costs 5/3× extra compute for marginal insight.
#
# - Feature 5 (Per-Fold State Save ✅): Each fold writes fold_metrics.json
#   immediately after training so partial results are recoverable if the session
#   times out. The CV summary JSON is written after all folds complete.
#
# How it works:
# - Step 1: Check USE_CV flag; skip if False.
# - Step 2: Load full_df (same file as Block 1.5, before split).
# - Step 3: Iterate StratifiedKFold splits, call run_fold() for each.
# - Step 4: Log mean ± std F1 across folds. Save CV summary JSON.
#
# Solutions attempted that did not work:
# - Running CV on train_df instead of full_df — excludes 20% of data from all folds,
#   producing pessimistically biased F1 estimates. Fixed by using full_df.
#
# Solutions implemented:
# - v1.0: CV on full_df with StratifiedKFold. Fresh model reload per fold.

print("======++++======")
print("Block 8 Output")
print("======++++======")
print()

import gc
import json
import logging
import os
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import DataCollatorForCompletionOnlyLM
from tqdm import tqdm

logger = logging.getLogger("dei_finetune")

USE_CV    = True
CV_EPOCHS = 3

if not USE_CV:
    logger.info(
        "Block 8 skipped (USE_CV=False). "
        "Set USE_CV=True only when single-split F1 std > 0.03 "
        "across ≥3 independent runs."
    )
else:
    CV_BASE_DIR = os.path.join(CHECKPOINT_DIR, "cv")
    os.makedirs(CV_BASE_DIR, exist_ok=True)

    # CV runs on the FULL dataset before the 80/20 split
    logger.info(f"Loading full dataset for CV from: {RAW_DATA_PATH}")
    df_full = pd.read_csv(RAW_DATA_PATH)
    df_full = df_full.rename(columns={"text": "Tweet_clean", "label": "DEI"})
    df_full = df_full.dropna(subset=["Tweet_clean", "DEI"]).reset_index(drop=True)
    df_full["DEI"] = df_full["DEI"].astype(int)
    logger.info(f"CV full dataset: {len(df_full):,} rows")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_f1_scores: list[float] = []

    def run_fold(
        fold_idx: int,
        train_idx: np.ndarray,
        val_idx: np.ndarray,
    ) -> float:
        logger.info(
            f"[Fold {fold_idx+1}/5] train={len(train_idx)} | val={len(val_idx)}"
        )
        fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        fold_train_texts = [
            format_prompt(r["Tweet_clean"], r["DEI"])
            for _, r in tqdm(
                fold_train.iterrows(), total=len(fold_train),
                desc=f"[Fold {fold_idx+1}] Format train", leave=False,
            )
        ]
        fold_val_texts = [
            format_prompt(r["Tweet_clean"], r["DEI"])
            for _, r in tqdm(
                fold_val.iterrows(), total=len(fold_val),
                desc=f"[Fold {fold_idx+1}] Format val", leave=False,
            )
        ]

        from datasets import Dataset as HFDataset
        fold_train_ds = HFDataset.from_dict({"text": fold_train_texts})
        fold_val_ds   = HFDataset.from_dict({"text": fold_val_texts})

        # Fresh model — prevents weight leakage across folds
        _bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        _base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=_bnb,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
        _base.config.use_cache      = False
        _base.config.pretraining_tp = 1

        _lora = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
            task_type=TaskType.CAUSAL_LM,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
        )
        _fold_model = get_peft_model(_base, _lora)
        _fold_model.config.use_cache = False
        _fold_model.enable_input_require_grads()

        _fold_collator = DataCollatorForCompletionOnlyLM(
            response_template=_response_template_ids,
            tokenizer=tokenizer,
            mlm=False,
        )
        _fold_output_dir = os.path.join(CV_BASE_DIR, f"fold_{fold_idx+1}")
        os.makedirs(_fold_output_dir, exist_ok=True)

        _fold_args = TrainingArguments(
            output_dir                  = _fold_output_dir,
            num_train_epochs            = CV_EPOCHS,
            per_device_train_batch_size = 4,
            gradient_accumulation_steps = 8,
            per_device_eval_batch_size  = 8,
            fp16=True, bf16=False,
            optim                       = "paged_adamw_8bit",
            gradient_checkpointing      = True,
            learning_rate               = 2e-4,
            weight_decay                = 0.01,
            lr_scheduler_type           = "cosine",
            warmup_ratio                = 0.10,
            eval_strategy               = "epoch",
            save_strategy               = "no",
            logging_steps               = 25,
            report_to                   = "none",
            seed                        = SEED + fold_idx,
            dataloader_num_workers      = 0,
            remove_unused_columns       = False,
            label_names                 = ["labels"],
        )

        _cw = compute_class_weight(
            class_weight="balanced",
            classes=np.array([0, 1]),
            y=fold_train["DEI"].values,
        )
        _cw = np.clip(_cw, 0.0, 5.0)
        _fold_cw = torch.tensor(_cw, dtype=torch.float32)

        _fold_compute_metrics = make_compute_metrics(label_token_ids)

        _fold_trainer = DEITrainer(
            model                         = _fold_model,
            args                          = _fold_args,
            train_dataset                 = fold_train_ds,
            eval_dataset                  = fold_val_ds,
            processing_class              = tokenizer,
            data_collator                 = _fold_collator,
            class_weights                 = _fold_cw,
            label_token_ids_map           = label_token_ids,
            compute_metrics               = _fold_compute_metrics,
            preprocess_logits_for_metrics = preprocess_logits_for_metrics,
        )

        _fold_trainer.train()
        _eval_results = _fold_trainer.evaluate()
        _fold_f1 = _eval_results.get("eval_f1_macro", 0.0)
        logger.info(f"[Fold {fold_idx+1}/5] F1 macro: {_fold_f1:.4f}")

        _fold_model.save_pretrained(_fold_output_dir)
        with open(
            os.path.join(_fold_output_dir, "fold_metrics.json"), "w"
        ) as _f:
            json.dump(
                {"fold": fold_idx + 1, "f1_macro": _fold_f1, "all": _eval_results},
                _f, indent=2,
            )

        del _fold_model, _fold_trainer, _base
        gc.collect()
        torch.cuda.empty_cache()
        return float(_fold_f1)

    for _fold_idx, (_tr_idx, _va_idx) in enumerate(
        tqdm(
            skf.split(df_full["Tweet_clean"], df_full["DEI"]),
            total=5,
            desc="Cross-validation folds",
        )
    ):
        _f1 = run_fold(_fold_idx, _tr_idx, _va_idx)
        fold_f1_scores.append(_f1)
        logger.info(
            f"[Fold {_fold_idx+1}/5] complete | F1: {_f1:.4f} | "
            f"Running mean: {np.mean(fold_f1_scores):.4f}"
        )

    _mean_f1 = float(np.mean(fold_f1_scores))
    _std_f1  = float(np.std(fold_f1_scores))

    logger.info("=" * 60)
    logger.info("CROSS-VALIDATION SUMMARY")
    logger.info("=" * 60)
    for _i, _s in enumerate(fold_f1_scores):
        logger.info(f"  Fold {_i+1}: F1 macro = {_s:.4f}")
    logger.info(f"  Mean F1 macro: {_mean_f1:.4f}")
    logger.info(f"  Std  F1 macro: {_std_f1:.4f}")

    if _std_f1 > 0.03:
        logger.warning(
            f"F1 std {_std_f1:.4f} > 0.03. High variance — "
            "audit 50 random samples per class for label noise."
        )
    else:
        logger.info(
            f"✅ F1 std {_std_f1:.4f} ≤ 0.03 — single-split result is reliable."
        )
    logger.info("=" * 60)

    _cv_summary = {
        "fold_f1_scores": fold_f1_scores,
        "mean_f1":        _mean_f1,
        "std_f1":         _std_f1,
        "cv_epochs":      CV_EPOCHS,
        "n_folds":        5,
        "source_file":    RAW_DATA_PATH,
        "note":           "CV run on full_df (pre-split), not train_df.",
    }
    with open(os.path.join(CV_BASE_DIR, "cv_summary.json"), "w") as _f:
        json.dump(_cv_summary, _f, indent=2)
    logger.info(f"CV summary saved to: {CV_BASE_DIR}/cv_summary.json")

# Overlap Analysis

In [ ]:
# ==========================
# Block OV-A: Tweet Overlap Analysis — Config & Total Counts
# Version: v1.0
# Purpose: Define folder configuration for overlap analysis, load all CSVs
#          from each configured folder, and display a detailed total tweet
#          count table per folder and per university using Rich.
# ==========================
#
# What this block does: (~360 words)
#
# - Feature 1 (Config Dict ✅): FOLDER_CONFIG is a single Python dict that
#   acts as the entire source of truth for this analysis. Each key is a
#   human-readable label (e.g. "F1", "F2", "F3"). Each value has: path
#   (absolute Drive path to the folder containing CSVs), tweet_col (column
#   name holding the raw tweet text), uni_col (column name for the university
#   identifier), and desc (freetext description). Adding a new folder means
#   adding one entry to this dict — no other code changes needed.
#
# - Feature 2 (Multi-file Load ✅): For each folder in FOLDER_CONFIG, the
#   block lists all .csv files, reads each with pd.read_csv (encoding_errors
#   replace to tolerate non-UTF8 content), concatenates them, and stores the
#   combined DataFrame under the folder label. Files missing the tweet_col or
#   uni_col are skipped with a logged warning — the rest of the files in that
#   folder still load. Each row is tagged with a source_file column for
#   debugging.
#
# - Feature 3 (Tweet Normalisation ✅): After loading, a normalised tweet
#   column `_tweet_norm` is computed: lowercase, strip leading/trailing
#   whitespace, collapse multiple spaces, remove URLs and @mentions. This
#   normalised form is used for matching in Block OV-B. The raw tweet is
#   preserved. Normalisation ensures "RT @uni: Hello World" and "hello world"
#   match correctly even if one dataset stripped RT prefixes.
#
# - Feature 4 (Rich Count Table ✅): Prints a per-folder, per-university
#   tweet count table using Rich. Columns: Folder, University, Tweet Count,
#   % of Folder Total. Subtotals per folder are shown in bold cyan. Grand
#   total at the bottom. This table is designed to be read before running
#   Block OV-B so the user knows the relative sizes going in.
#
# How it works:
# - Step 1: Define FOLDER_CONFIG.
# - Step 2: For each folder, glob *.csv, read and concat.
# - Step 3: Compute _tweet_norm on each combined DataFrame.
# - Step 4: Build and print the Rich count table.
#
# Solutions attempted that did not work:
# - Hard-coding folder paths inline in the analysis code — breaks when a new
#   folder is added. Fixed by centralising everything in FOLDER_CONFIG.
# - Using tweet text as-is for matching — minor whitespace/case differences
#   cause false negatives. Fixed by computing _tweet_norm before matching.
#
# Solutions implemented:
# - Config-dict driven loading. Tweet normalisation pre-computed in this block
#   so Block OV-B can match directly without re-normalising.
# ==========================

print("======++++======")
print("Block OV-A Output")
print("======++++======")
print()

import os, re, logging
import pandas as pd
from rich.console import Console
from rich.table   import Table
from rich.text    import Text

logger  = logging.getLogger("dei_finetune")
console = Console(width=140)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit this dict to add / remove folders. No other code changes needed.
# ══════════════════════════════════════════════════════════════════════════════
FOLDER_CONFIG: dict[str, dict] = {
    "F1": {
        "path":      "/content/drive/MyDrive/Twitter/Finetune/inference/input/",
        "tweet_col": "Tweet",
        "uni_col":   "University",
        "desc":      "Main",
    },
    "F2": {
        "path":      "/content/drive/MyDrive/Twitter/Finetune/inference/input/Engineering/",
        "tweet_col": "Tweet",
        "uni_col":   "University",
        "desc":      "Engineering",
    }
    # "F3": {
    #     "path":      "/content/drive/MyDrive/Twitter/New_everything/Colab/Files/gemini-3-flash-preview/merge",
    #     "tweet_col": "Tweet",
    #     "uni_col":   "University",
    #     "desc":      "Gemini 3 Flash Preview output",
    # },
    # Add more folders here, e.g.:
    # "F4": {
    #     "path":      "/content/drive/MyDrive/...",
    #     "tweet_col": "Tweet",
    #     "uni_col":   "University",
    #     "desc":      "Another model output",
    # },
}

# Folder to check AGAINST — Block OV-B asks "are F2/F3 tweets inside F1?"
REFERENCE_KEY = "F1"

# ── Normalisation helper ───────────────────────────────────────────────────
_RE_URL     = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_RE_MENTION = re.compile(r"@\w+")
_RE_RT      = re.compile(r"^RT\s+", re.IGNORECASE)
_RE_WS      = re.compile(r"\s+")

def _normalise(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = _RE_RT.sub("",   text)
    text = _RE_URL.sub(" ", text)
    text = _RE_MENTION.sub(" ", text)
    text = _RE_WS.sub(" ", text).strip().lower()
    return text

# ── Load all folders ───────────────────────────────────────────────────────
LOADED: dict[str, pd.DataFrame] = {}

for label, cfg in FOLDER_CONFIG.items():
    folder_path = cfg["path"]
    tweet_col   = cfg["tweet_col"]
    uni_col     = cfg["uni_col"]

    if not os.path.isdir(folder_path):
        logger.warning(f"[{label}] Folder not found: {folder_path} — skipping.")
        LOADED[label] = pd.DataFrame()
        continue

    csv_files = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(".csv"))
    if not csv_files:
        logger.warning(f"[{label}] No CSV files in: {folder_path}")
        LOADED[label] = pd.DataFrame()
        continue

    frames = []
    for fname in csv_files:
        fpath = os.path.join(folder_path, fname)
        try:
            df = pd.read_csv(fpath, encoding="utf-8", encoding_errors="replace",
                             on_bad_lines="skip")
        except Exception as _e:
            logger.warning(f"  [{label}] Could not read {fname}: {_e}")
            continue
        if tweet_col not in df.columns:
            logger.warning(f"  [{label}] {fname}: missing column '{tweet_col}' — skipping file.")
            continue
        if uni_col not in df.columns:
            df[uni_col] = "Unknown"
        df["_source_file"] = fname
        frames.append(df[[tweet_col, uni_col, "_source_file"]])

    if not frames:
        logger.warning(f"[{label}] All files skipped — no valid data.")
        LOADED[label] = pd.DataFrame()
        continue

    combined = pd.concat(frames, ignore_index=True)
    combined["_tweet_norm"] = combined[tweet_col].apply(_normalise)
    # Drop completely empty normalised tweets
    combined = combined[combined["_tweet_norm"].str.len() > 0].reset_index(drop=True)
    LOADED[label] = combined
    logger.info(f"[{label}] Loaded {len(combined):,} tweets from {len(frames)} file(s).")

# ── Rich Count Table ───────────────────────────────────────────────────────
tbl = Table(
    title="Tweet Count by Folder & University",
    show_header=True, header_style="bold magenta", show_lines=False,
)
tbl.add_column("Folder",      style="bold cyan",  min_width=8)
tbl.add_column("Description", style="dim white",  min_width=35)
tbl.add_column("University",  style="white",       min_width=30)
tbl.add_column("Count",       justify="right",     min_width=8)
tbl.add_column("% of Folder", justify="right",     min_width=10)

grand_total = 0

for label, cfg in FOLDER_CONFIG.items():
    df = LOADED.get(label, pd.DataFrame())
    tweet_col = cfg["tweet_col"]
    uni_col   = cfg["tweet_col"]   # intentional: reuse key names from cfg
    uni_col   = cfg["uni_col"]

    if df.empty:
        tbl.add_row(label, cfg["desc"], "—", "0", "—")
        tbl.add_section()
        continue

    folder_total = len(df)
    grand_total += folder_total
    uni_counts   = df[uni_col].value_counts().sort_index()
    first_row    = True

    for uni, cnt in uni_counts.items():
        pct = cnt / folder_total * 100
        tbl.add_row(
            label if first_row else "",
            cfg["desc"] if first_row else "",
            str(uni),
            f"{cnt:,}",
            f"{pct:.1f}%",
        )
        first_row = False

    # Subtotal row
    tbl.add_row(
        "", "", Text(f"SUBTOTAL ({label})", style="bold"),
        Text(f"{folder_total:,}", style="bold cyan"),
        Text("100.0%", style="bold cyan"),
    )
    tbl.add_section()

# Grand total
tbl.add_row(
    Text("ALL", style="bold yellow"),
    Text("Grand Total", style="bold yellow"),
    "",
    Text(f"{grand_total:,}", style="bold yellow"),
    "",
)
console.print(tbl)
logger.info(f"OV-A complete. Grand total tweets across all folders: {grand_total:,}")
logger.info("✅ Block OV-A ready. Run Block OV-B for overlap analysis.")

In [ ]:
# ==========================
# Block OV-B: Tweet Overlap Analysis — Percentage Overlap (F2/F3 vs F1)
# Version: v1.0
# Purpose: For every non-reference folder, compute exact and normalised-match
#          overlap against the reference folder (F1), broken down by university,
#          and display results in a Rich table with per-university and overall
#          overlap percentages.
# ==========================
#
# What this block does: (~390 words)
#
# - Feature 1 (Reference Set Construction ✅): Builds a Python set of normalised
#   tweet strings from LOADED[REFERENCE_KEY]. Set lookups are O(1) regardless
#   of corpus size. For F1 with ~200k tweets, set construction takes ~1 second
#   and membership testing for an entire F2/F3 corpus takes under 2 seconds.
#   This is far faster than a DataFrame merge (which requires sorting or hashing
#   both sides) and requires no join keys.
#
# - Feature 2 (Per-University Breakdown ✅): For each comparison folder, the
#   overlap is computed separately per university label so the user can see
#   which universities have high or low overlap. A university with 95% overlap
#   means nearly all its tweets in F2 also exist in F1. A university with 5%
#   overlap means the two datasets are almost completely disjoint for that
#   university.
#
# - Feature 3 (Directional Clarity ✅): The question asked is "does F1 contain
#   tweets from F2/F3?" — i.e. we check each F2/F3 tweet against F1 (not the
#   reverse). The overlap percentage is reported as "% of F2 tweets found in F1"
#   — the denominator is the comparison folder, not the reference. A second
#   metric "% of F1 covered by F2" (reverse direction) is also computed for
#   completeness. Both are shown in the table.
#
# - Feature 4 (Rich Table ✅): One table per comparison folder, with columns:
#   University, F2 Tweets, Found in F1, % F2→F1, F1 Total (for that uni),
#   % F1 covered. Colour-coded: ≥50% overlap in green, 10-49% in yellow,
#   <10% in red. Subtotal row per table.
#
# - Feature 5 (Summary Log ✅): After all tables, logs a one-liner summary
#   per comparison folder: overall overlap % in both directions.
#
# How it works:
# - Step 1: Build ref_set from LOADED[REFERENCE_KEY]["_tweet_norm"].
# - Step 2: For each non-reference label in FOLDER_CONFIG: iterate universities,
#           compute overlap counts, collect rows.
# - Step 3: Print Rich table per comparison folder.
# - Step 4: Log summary.
#
# Solutions attempted that did not work:
# - pd.merge on raw tweet text — case/whitespace differences cause false
#   negatives. Fixed by using _tweet_norm (pre-computed in OV-A).
# - pd.merge without per-university breakdown — hides which universities
#   drive the overlap. Fixed by iterating per university.
#
# Solutions implemented:
# - Set-based lookup on _tweet_norm. Per-university breakdown. Bidirectional
#   overlap percentages. Colour-coded Rich table.
# ==========================

print("======++++======")
print("Block OV-B Output")
print("======++++======")
print()

import logging
import pandas as pd
from rich.console import Console
from rich.table   import Table
from rich.text    import Text

logger  = logging.getLogger("dei_finetune")
console = Console(width=160)

# ── Verify OV-A has run ────────────────────────────────────────────────────
if "LOADED" not in dir() or not LOADED:
    raise RuntimeError(
        "LOADED dict not found. Run Block OV-A first to load the tweet data."
    )
if REFERENCE_KEY not in LOADED or LOADED[REFERENCE_KEY].empty:
    raise RuntimeError(
        f"Reference folder '{REFERENCE_KEY}' is empty or missing. "
        "Check FOLDER_CONFIG in Block OV-A and ensure the folder path is correct."
    )

# ── Build reference set ────────────────────────────────────────────────────
ref_df  = LOADED[REFERENCE_KEY]
ref_uni = FOLDER_CONFIG[REFERENCE_KEY]["uni_col"]

ref_set: set[str] = set(ref_df["_tweet_norm"].tolist())
logger.info(
    f"Reference set built from '{REFERENCE_KEY}': "
    f"{len(ref_set):,} unique normalised tweets."
)

# Pre-index reference by university for reverse-direction coverage
ref_by_uni: dict[str, set[str]] = {}
for uni, grp in ref_df.groupby(ref_uni):
    ref_by_uni[str(uni)] = set(grp["_tweet_norm"].tolist())

# ── Per-folder overlap analysis ────────────────────────────────────────────
comparison_keys = [k for k in FOLDER_CONFIG if k != REFERENCE_KEY]

for comp_key in comparison_keys:
    comp_df = LOADED.get(comp_key, pd.DataFrame())

    if comp_df.empty:
        logger.warning(f"[{comp_key}] No data loaded — skipping overlap analysis.")
        continue

    comp_uni_col = FOLDER_CONFIG[comp_key]["uni_col"]

    # ── Build Rich table ────────────────────────────────────────────────────
    tbl = Table(
        title=(
            f"Overlap: {comp_key} → {REFERENCE_KEY}  "
            f"('{FOLDER_CONFIG[comp_key]['desc']}' vs "
            f"'{FOLDER_CONFIG[REFERENCE_KEY]['desc']}')"
        ),
        show_header=True, header_style="bold magenta", show_lines=False,
    )
    tbl.add_column("University",         style="white",       min_width=32)
    tbl.add_column(f"{comp_key} Tweets", justify="right",     min_width=12)
    tbl.add_column(f"Found in {REFERENCE_KEY}", justify="right", min_width=14)
    tbl.add_column(f"% {comp_key}→{REFERENCE_KEY}", justify="right", min_width=16)
    tbl.add_column(f"{REFERENCE_KEY} Total (uni)", justify="right", min_width=16)
    tbl.add_column(f"% {REFERENCE_KEY} covered",  justify="right", min_width=16)

    total_comp      = 0
    total_found     = 0

    # Compute per-university rows
    rows: list[tuple] = []
    for uni, grp in comp_df.groupby(comp_uni_col):
        uni_str    = str(uni)
        comp_count = len(grp)
        found      = grp["_tweet_norm"].isin(ref_set).sum()
        pct_fwd    = found / comp_count * 100 if comp_count > 0 else 0.0

        ref_uni_set   = ref_by_uni.get(uni_str, set())
        ref_uni_count = len(ref_uni_set)
        pct_rev       = found / ref_uni_count * 100 if ref_uni_count > 0 else 0.0

        total_comp  += comp_count
        total_found += found
        rows.append((uni_str, comp_count, found, pct_fwd, ref_uni_count, pct_rev))

    # Sort by forward overlap % descending for readability
    rows.sort(key=lambda r: r[3], reverse=True)

    def _pct_style(pct: float) -> str:
        if pct >= 50: return "bold bright_green"
        if pct >= 10: return "bold yellow"
        return "bold red"

    for uni_str, comp_count, found, pct_fwd, ref_uni_count, pct_rev in rows:
        tbl.add_row(
            uni_str,
            f"{comp_count:,}",
            f"{found:,}",
            Text(f"{pct_fwd:.1f}%", style=_pct_style(pct_fwd)),
            f"{ref_uni_count:,}",
            Text(f"{pct_rev:.1f}%", style=_pct_style(pct_rev)),
        )

    # Subtotal
    overall_pct_fwd = total_found / total_comp * 100 if total_comp > 0 else 0.0
    ref_total       = len(ref_set)
    overall_pct_rev = total_found / ref_total * 100 if ref_total > 0 else 0.0

    tbl.add_section()
    tbl.add_row(
        Text("OVERALL", style="bold"),
        Text(f"{total_comp:,}",  style="bold cyan"),
        Text(f"{total_found:,}", style="bold cyan"),
        Text(f"{overall_pct_fwd:.2f}%", style=_pct_style(overall_pct_fwd)),
        Text(f"{ref_total:,}",  style="bold"),
        Text(f"{overall_pct_rev:.2f}%", style=_pct_style(overall_pct_rev)),
    )

    console.print(tbl)
    console.print()

    logger.info(
        f"[{comp_key} → {REFERENCE_KEY}] "
        f"{total_found:,}/{total_comp:,} tweets found in {REFERENCE_KEY} "
        f"({overall_pct_fwd:.2f}% of {comp_key} exists in {REFERENCE_KEY}). "
        f"Reference coverage: {overall_pct_rev:.2f}% of {REFERENCE_KEY} is covered by {comp_key}."
    )

console.print(
    "\n[bold]Interpretation guide:[/bold]\n"
    "  [bold bright_green]≥50%[/bold bright_green] — significant overlap, datasets likely share source.\n"
    "  [bold yellow]10–49%[/bold yellow] — partial overlap, some shared tweets.\n"
    "  [bold red]<10%[/bold red]  — datasets are largely disjoint.\n"
    "\n"
    f"  '% {comparison_keys[0]}→{REFERENCE_KEY}' = what fraction of {comparison_keys[0]} tweets appear in {REFERENCE_KEY}.\n"
    f"  '% {REFERENCE_KEY} covered' = what fraction of {REFERENCE_KEY} is contributed by {comparison_keys[0]}.\n"
)
logger.info("✅ Block OV-B complete.")

# Block 11

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 11a — Imports · Config · UI Helpers · Prompt/Cleaning utils
# Version: v2.0
# Fixes:
#   - _vram_str() now shows USED/TOTAL (not free/total — matches nvidia-smi)
#   - BATCH_SIZE_HIGH raised to 128 (safe after lm_head bypass in 11d)
#   - stat_line supports optional bullet for sub-items (Processed Files row)
# ════════════════════════════════════════════════════════════════════════════
import gc, os, re, time
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from rich.console import Console
from rich.live    import Live
from rich.table   import Table
from rich.text    import Text

drive.mount("/content/drive")

# ── CONFIG ────────────────────────────────────────────────────────────────────
BASE_DIR             = "/content/drive/MyDrive/Twitter/Finetune"
# MODEL_NAME           = "Qwen/Qwen3.5-9B"
# ADAPTER_DIR          = os.path.join(BASE_DIR, "checkpoints", "qwen3.5_4b", "best_model")
# INFERENCE_INPUT_DIR  = os.path.join(BASE_DIR, "inference", "input")
# INFERENCE_OUTPUT_DIR = os.path.join(BASE_DIR, "inference", "output", "qwen3.5_4b")



MODEL_NAME           = "Qwen/Qwen3.5-4B"
ADAPTER_DIR          = os.path.join(BASE_DIR, "checkpoints", "qwen3.5_4b", "best_model")
INFERENCE_INPUT_DIR  = os.path.join(BASE_DIR, "inference", "input")
INFERENCE_OUTPUT_DIR = os.path.join(BASE_DIR, "inference", "output", "qwen3.5_4b")

os.makedirs(INFERENCE_INPUT_DIR,  exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

INFERENCE_THRESHOLD = 0.72
BATCH_SIZE_HIGH     = 128   # raised from 64 — safe after lm_head bypass memory fix
BATCH_SIZE_LOW      = 32    # OOM fallback

torch.backends.cudnn.benchmark = True
console = Console(width=160)

# ── UI HELPERS ────────────────────────────────────────────────────────────────
def _fmt_time(s: float) -> str:
    if s < 0 or s != s: return "?"
    h, rem = divmod(int(s), 3600)
    m, sec = divmod(rem, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

def _bar(pct: float, width: int = 20, done: bool = False, active: bool = False) -> Text:
    """bright_green=done · cyan=active · white=pending"""
    filled = max(0, min(width, int(width * pct / 100)))
    color  = "bright_green" if done else ("cyan" if active else "grey50")
    t = Text()
    t.append("█" * filled,           style=color)
    t.append("░" * (width - filled), style="grey35")
    return t

def _vram_str() -> str:
    """
    Returns USED/TOTAL GB — matches what nvidia-smi reports as 'in use'.
    memory_reserved() = memory held by the PyTorch CUDA allocator (model +
    activation pool). This is what you see in nvidia-smi, not memory_allocated()
    which only counts live tensors.
    """
    used  = torch.cuda.memory_reserved(0)  / 1024 ** 3
    total = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    return f"{used:.1f}/{total:.1f} GB"

# ── PROMPT & CLEANING ─────────────────────────────────────────────────────────
# _SYSTEM = (
#     "You are a binary classifier. Reply with only the digit 0 or 1. "
#     "No explanation. No reasoning. /no_think"
# )
_SYSTEM = (
    "You are a binary classifier identifying Diversity, Equity, and Inclusion (DEI) initiatives in university tweets. "
    "Reply with 1 if the tweet explicitly discusses structural equity, inclusion, diversity initiatives, or accessibility. "
    "Reply with 0 if it is a general campus update, cultural holiday greeting, biomedical research, standard marketing or things that are not related to DEI. "
    "Reply with only the digit 0 or 1. No explanation. No reasoning. /no_think"
)


def format_inference_prompt(tweet: str) -> str:
    return (
        f"<|im_start|>system\n{_SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\nTweet: {tweet}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

_RE_URL     = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_RE_MENTION = re.compile(r"@\w+")
_RE_HASHTAG = re.compile(r"#(\w+)")
_RE_WS      = re.compile(r"\s+")

def clean_tweet(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower()
    text = _RE_URL.sub(" ", text)
    text = _RE_MENTION.sub(" ", text)
    text = _RE_HASHTAG.sub(r"\1", text)
    return _RE_WS.sub(" ", text).strip()

console.print("✅ Block 11a ready.", style="bold green")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ Block 11a ready.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 11b — Load Tokenizer & Model
# Version: v2.0
# Fixes / Additions:
#   - Pre-fetches _lm_head_w2: only the 2 token rows from lm_head weight matrix.
#     This enables the full vocab projection bypass in Block 11d, reducing
#     peak VRAM from ~14 GB → ~6-7 GB per batch, unlocking batch_size=128.
#   - Verifies the bypass path (inf_model.base_model.model.model) at load time.
#     If path fails, warns clearly with the diagnostic command to check.
#   - VRAM display now shows USED (not free) to match nvidia-smi.
# ════════════════════════════════════════════════════════════════════════════
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

console.print("📝 Loading tokenizer...", style="bold")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

T0_ID = tokenizer("0", add_special_tokens=False)["input_ids"][0]
T1_ID = tokenizer("1", add_special_tokens=False)["input_ids"][0]
console.print(f"   Token IDs — '0': {T0_ID} | '1': {T1_ID}", style="dim")

_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

if not globals().get("INF_MODEL_LOADED", False):
    for _k in ["model", "inf_model", "_lm_head_w2"]:
        if _k in globals():
            del globals()[_k]
    gc.collect()
    torch.cuda.empty_cache()

    console.print("Loading base model in 4-bit from HF cache...", style="bold")
    _t0   = time.time()
    _base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=_bnb_cfg,
        device_map={"": 0},
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    console.print(f"Base loaded in {time.time() - _t0:.0f}s", style="green")

    console.print("Loading adapter from Drive...", style="bold")
    _t1       = time.time()
    inf_model = PeftModel.from_pretrained(_base, ADAPTER_DIR)
    inf_model.eval()
    console.print(f"Adapter loaded in {time.time() - _t1:.0f}s", style="green")

    # ── Pre-fetch lm_head weight slice ────────────────────────────────────────
    # Normal inference: inf_model(**enc) → logits (B, S, 151936) → ~10 GB peak
    # Bypass:  transformer body → last_hidden (B, D) → matmul with 2 weight rows → (B, 2)
    # Peak VRAM drops from ~14 GB to ~6-7 GB, making batch_size=128 safe on T4.
    try:
        _lm_head_w2 = (
            inf_model.base_model.model.lm_head.weight[[T0_ID, T1_ID], :]
            .detach().clone()
        )  # shape: (2, hidden_dim) — stays on GPU, ~10 KB
        console.print(
            f"✅ lm_head bypass ready — weight slice shape: {list(_lm_head_w2.shape)}",
            style="dim green",
        )
        globals()["_LM_HEAD_BYPASS_OK"] = True
    except AttributeError as _lm_err:
        _lm_head_w2 = None
        console.print(
            f"⚠️  lm_head bypass path failed: {_lm_err}\n"
            "   Falling back to full forward pass (higher VRAM, smaller batch).\n"
            "   Diagnostic: print(type(inf_model.base_model.model))",
            style="yellow",
        )
        globals()["_LM_HEAD_BYPASS_OK"] = False

    console.print(f"VRAM after load: {_vram_str()}", style="cyan")

    globals()["INF_MODEL_LOADED"] = True
    globals()["inf_model"]        = inf_model
    globals()["_lm_head_w2"]      = _lm_head_w2

else:
    inf_model   = globals()["inf_model"]
    _lm_head_w2 = globals()["_lm_head_w2"]
    console.print(f"✅ Model already in memory.  VRAM: {_vram_str()}", style="green")

console.print("✅ Block 11b ready.", style="bold green")

📝 Loading tokenizer...

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading adapter from Drive...

Adapter loaded in 5s

✅ lm_head bypass ready — weight slice shape: [2, 2560]

VRAM after load: 3.2/14.6 GB

✅ Block 11b ready.

In [ ]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR            = "/content/drive/MyDrive/Twitter/Finetune"
INFERENCE_INPUT_DIR = os.path.join(BASE_DIR, "inference", "input")

CATEGORY_DIRS = {
    "Main":        os.path.join(INFERENCE_INPUT_DIR, "Main"),       # ← changed
    "Engineering": os.path.join(INFERENCE_INPUT_DIR, "Engineering"),
    "Law":         os.path.join(INFERENCE_INPUT_DIR, "Law"),
    "Medical":     os.path.join(INFERENCE_INPUT_DIR, "Med"),        # ← your folder is "Med" not "Medical"
    "Business":    os.path.join(INFERENCE_INPUT_DIR, "Business"),
}

print(f"{'Category':<15} {'Files':>6} {'Rows':>12}")
print("─" * 38)
grand_total = 0

for cat, folder in CATEGORY_DIRS.items():
    if not os.path.isdir(folder):
        print(f"{cat:<15} {'—':>6} {'folder not found':>12}")
        continue

    # Only direct CSVs — skip subdirectories
    csv_files = [
        f for f in os.listdir(folder)
        if f.lower().endswith(".csv")
        and os.path.isfile(os.path.join(folder, f))
    ]

    total_rows = 0
    for fname in csv_files:
        try:
            df = pd.read_csv(
                os.path.join(folder, fname),
                usecols=["University"] if True else None,
                encoding_errors="replace",
            )
            total_rows += len(df)
        except Exception as e:
            print(f"  Warning {fname}: {e}")

    grand_total += total_rows
    print(f"{cat:<15} {len(csv_files):>6} {total_rows:>12,}")

print("─" * 38)
print(f"{'TOTAL':<15} {'':>6} {grand_total:>12,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Category         Files         Rows
──────────────────────────────────────
Main                20      338,833
Engineering         51      216,239
Law                 38      279,061
Medical             41      204,126
Business            27      306,687
──────────────────────────────────────
TOTAL                     1,344,946


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 11c — Scan Input Files
# Version: v2.0
# Fixes:
#   - CRITICAL: start_idx capped at total_rows (min(len(done_df), total_rows)).
#     Without this, output CSVs with duplicate rows from previous partial runs
#     produce start_idx > total_rows → negative Remaining → grand_total > total.
#   - Added prior_done flag on each task (True = fully complete before this session).
#   - prior_done_count exposed for Block 11d header "Processed Files" display.
#   - Scan output prints all relevant counts including pending file count.
# ════════════════════════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 11c v2.0 — Scan Input Files (Multi-Category)
# ════════════════════════════════════════════════════════════════════════════
# import console
from rich.console import Console
console = Console(width=160)


CATEGORY_DIRS = {
    "Main":        os.path.join(INFERENCE_INPUT_DIR, "Main"),       # ← changed
    "Engineering": os.path.join(INFERENCE_INPUT_DIR, "Engineering"),
    "Law":         os.path.join(INFERENCE_INPUT_DIR, "Law"),
    "Medical":     os.path.join(INFERENCE_INPUT_DIR, "Med"),        # ← your folder is "Med" not "Medical"
    "Business":    os.path.join(INFERENCE_INPUT_DIR, "Business"),
}

# Category-specific output dirs
# ── Define model output base explicitly ────────────────────────────────────
_MODEL_OUTPUT_BASE = os.path.join(BASE_DIR, "inference", "output", "qwen3.5_4b")

CATEGORY_OUTPUT_DIRS = {
    "Main":        os.path.join(_MODEL_OUTPUT_BASE, "Main"),
    "Engineering": os.path.join(_MODEL_OUTPUT_BASE, "Engineering"),
    "Law":         os.path.join(_MODEL_OUTPUT_BASE, "Law"),
    "Medical":     os.path.join(_MODEL_OUTPUT_BASE, "Medical"),
    "Business":    os.path.join(_MODEL_OUTPUT_BASE, "Business"),
}

for _d in CATEGORY_OUTPUT_DIRS.values():
    os.makedirs(_d, exist_ok=True)

console.print("🔍 Scanning files across all categories...", style="bold")

file_tasks:        list[dict] = []
overall_total:     int = 0
overall_processed: int = 0
prior_done_count:  int = 0

for category, input_dir in CATEGORY_DIRS.items():
    if not os.path.isdir(input_dir):
        console.print(f"   ⚠ [{category}] folder not found: {input_dir}", style="yellow")
        continue

    output_dir = CATEGORY_OUTPUT_DIRS[category]

    # Only direct CSV files — skip subdirectories inside this folder
    input_files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".csv")
        and os.path.isfile(os.path.join(input_dir, f))
    )

    if not input_files:
        console.print(f"   ⚠ [{category}] no CSV files found", style="yellow")
        continue

    console.print(f"   [{category}] {len(input_files)} file(s)", style="dim")

    for fname in input_files:
        input_path  = os.path.join(input_dir, fname)
        output_path = os.path.join(output_dir, f"classified_{fname}")

        try:
            df = pd.read_csv(input_path, encoding="utf-8", encoding_errors="replace")
        except Exception as _e:
            console.print(f"   ⚠ Skipping {fname}: {_e}", style="yellow")
            continue

        if "Tweet" not in df.columns:
            console.print(f"   ⚠ Skipping {fname}: no 'Tweet' column", style="yellow")
            continue

        df["char_len"] = df["Tweet"].astype(str).str.len()
        df["orig_idx"] = df.index
        df_sorted      = df.sort_values("char_len").reset_index(drop=True)
        total_rows     = len(df_sorted)
        overall_total += total_rows

        start_idx = 0
        cur_dei_1 = 0
        cur_dei_0 = 0

        if os.path.isfile(output_path):
            try:
                done_df   = pd.read_csv(output_path, on_bad_lines="skip")
                raw_count = len(done_df)
                start_idx = min(raw_count, total_rows)
                if raw_count > total_rows:
                    console.print(
                        f"   ⚠ {fname}: output {raw_count} > input {total_rows}. Capped.",
                        style="yellow",
                    )
                if "dei_label" in done_df.columns:
                    _valid    = done_df.head(total_rows)
                    cur_dei_1 = int((_valid["dei_label"] == 1).sum())
                    cur_dei_0 = int((_valid["dei_label"] == 0).sum())
            except pd.errors.EmptyDataError:
                pass
            except Exception as _scan_err:
                console.print(f"   ⚠ Could not read output {fname}: {_scan_err}", style="yellow")

        overall_processed += start_idx
        already_done       = (start_idx >= total_rows)
        if already_done:
            prior_done_count += 1

        file_tasks.append({
            "fname":       fname,
            "category":    category,           # NEW
            "df":          df_sorted,
            "output_path": output_path,        # already category-specific
            "total_rows":  total_rows,
            "start_idx":   start_idx,
            "cur_dei_1":   cur_dei_1,
            "cur_dei_0":   cur_dei_0,
            "prior_done":  already_done,
        })

pending_count = len(file_tasks) - prior_done_count
console.print(
    f"   • Total files      : {len(file_tasks)}\n"
    f"   • Already complete : {prior_done_count}\n"
    f"   • Pending          : {pending_count}\n"
    f"   • Total tweets     : {overall_total:,}\n"
    f"   • Already done     : {overall_processed:,}\n"
    f"   • Remaining        : {overall_total - overall_processed:,}",
    style="yellow",
)
console.print("✅ Block 11c v2.0 ready.", style="bold green")

🔍 Scanning files across all categories...

   [Main] 20 file(s)

   [Engineering] 51 file(s)

   ⚠ 10_DukeEngineering_engg_2010_2022.csv: output 3652 > input 1923. Capped.

   [Law] 38 file(s)

   [Medical] 41 file(s)

   [Business] 27 file(s)

   • Total files      : 177
   • Already complete : 136
   • Pending          : 41
   • Total tweets     : 1,344,946
   • Already done     : 1,129,914
   • Remaining        : 215,032

✅ Block 11c v2.0 ready.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 11d — Inference Loop · Live UI · Session Summary
# Version: v2.3
# Fixes / Changes:
#   - AUTO-MAXIMIZING VRAM: Starts with an aggressive batch size (384) to
#     intentionally push VRAM usage to 90%+.
#   - SMART OOM CATCHER: Instead of dropping to a hard limit like 16, it scales
#     down by 20% on OOM to find the exact maximum batch size your GPU can handle.
#   - Processed Files display perfectly matches standard bulleted line style.
#   - Speed display includes tweets/min.
# ════════════════════════════════════════════════════════════════════════════

# ── Dynamic Batch Sizing Configuration ─────────────────────────────────────────
# Start very high. The new OOM logic will dial this down perfectly to fit ~90% VRAM.
STARTING_BATCH_SIZE = 256

# ── Build task → file_stats index mapping (prior-done files are excluded) ──────
task_to_stat_idx: dict[int, int] = {}
_stat_i = 0
for _ti, _t in enumerate(file_tasks):
    if not _t["prior_done"]:
        task_to_stat_idx[_ti] = _stat_i
        _stat_i += 1

file_stats: list[dict] = []
for _t in file_tasks:
    if _t["prior_done"]:
        continue
    file_stats.append({
        "fname":     _t["fname"],
        "total":     _t["total_rows"],
        "processed": _t["start_idx"],
        "dei":       _t["cur_dei_1"],
        "non_dei":   _t["cur_dei_0"],
        "fail":      0,
        "done":      False,
        "t_start":   None,
        "t_end":     None,
    })

# ── _build_display ─────────────────────────────────────────────────────────────
def _build_display(
    file_stats:        list,
    overall:           dict,
    active_stat_idx:   int,
    session_start:     float,
    visible_count:     int,
    current_batch_size:int,
    oom_downgraded:    bool,
    tweets_per_sec:    float,
    prior_done_count:  int,
    total_file_count:  int,
) -> Table:
    elapsed   = time.time() - session_start
    ov_proc   = overall["processed"]
    ov_tot    = overall["total"]
    remaining = max(0, ov_tot - ov_proc)
    ov_pct    = min(100.0, ov_proc / max(ov_tot, 1) * 100)
    ov_done   = (remaining == 0)

    # ETA
    if tweets_per_sec > 0:
        eff_tps = tweets_per_sec
    else:
        sess_proc = overall.get("session_processed", 0)
        eff_tps   = sess_proc / max(elapsed, 1e-9)
    ov_eta = remaining / max(eff_tps, 1e-9)

    # Speed strings
    tps = tweets_per_sec if tweets_per_sec > 0 else (
        overall.get("session_processed", 0) / max(elapsed, 1e-9)
    )
    tpm = tps * 60
    tph = tps * 3600
    if overall.get("session_processed", 0) > 10:
        speed_str = f"{tps:.1f} tweets/sec  |  {tpm:,.0f} tweets/min  |  {tph:,.0f} tweets/hr"
    else:
        speed_str = "calculating..."

    files_done_count = prior_done_count + overall.get("completed_files", 0)

    root = Table(box=None, show_header=False, padding=(0, 0), expand=True)
    root.add_column("c", no_wrap=True)
    sep  = Text("═" * 120, style="grey30")
    sep2 = Text("─" * 120, style="grey23")

    def stat_line(label: str, value: str, val_style: str = "white",
                  bullet: bool = True) -> Text:
        prefix = "  • " if bullet else "    "
        t = Text(no_wrap=True)
        t.append(f"{prefix}{label:<24}", style="dim white")
        t.append(value, style=val_style)
        return t

    # ── Stats header ───────────────────────────────────────────────────────────
    root.add_row(sep2)
    root.add_row(stat_line("Total Files", str(total_file_count)))
    root.add_row(stat_line(
        "Processed Files",
        f"{files_done_count}",
        val_style="bright_green" if files_done_count == total_file_count else "white",
    ))
    root.add_row(stat_line("Total Tweets",      f"{ov_tot:,}"))
    root.add_row(stat_line("Already Processed", f"{ov_proc:,}"))
    root.add_row(stat_line(
        "Remaining",
        f"{remaining:,}",
        val_style="bright_green" if remaining == 0 else "bright_yellow",
    ))
    root.add_row(stat_line(
        "Batch Size",
        f"{current_batch_size}" + ("  (Auto-Tuned via OOM)" if oom_downgraded else "  (Pushing limits...)"),
        val_style="yellow" if oom_downgraded else "white",
    ))
    root.add_row(stat_line("Speed",  speed_str, val_style="bright_cyan"))
    root.add_row(stat_line("VRAM",   _vram_str(), val_style="cyan"))
    root.add_row(sep2)

    # ── Overall progress bar ───────────────────────────────────────────────────
    ov = Text(no_wrap=True)
    ov.append("OVERALL PROGRESS", style="bold yellow")
    ov.append(f" | DEI: {overall['dei']:,}",         style="bold bright_green")
    ov.append(f" | Non-DEI: {overall['non_dei']:,}",  style="bold white")
    ov.append(
        f" | Fail: {overall['fail']}",
        style="bold red" if overall["fail"] else "bold white",
    )
    ov.append(f" | {ov_pct:3.0f}%  ", style="bold white")
    ov.append_text(_bar(ov_pct, width=20, done=ov_done, active=not ov_done))
    ov.append(
        f"  {ov_proc:,}/{ov_tot:,}  [{_fmt_time(elapsed)}<{_fmt_time(ov_eta)}]",
        style="dim",
    )
    root.add_row(sep)
    root.add_row(ov)
    root.add_row(sep)

    # ── Per-file rows — ONLY pending files, lazy reveal ────────────────────────
    for i, fs in enumerate(file_stats[:visible_count]):
        proc   = fs["processed"]
        total  = fs["total"]
        pct    = min(100.0, proc / max(total, 1) * 100)
        done   = fs["done"]
        active = (i == active_stat_idx) and not done

        fname = (fs["fname"][:28] + "…") if len(fs["fname"]) > 29 else fs["fname"]

        if fs.get("t_start") is None:
            ftime = "?<?"
        elif done and fs.get("t_end") is not None:
            fe    = fs["t_end"] - fs["t_start"]
            ftime = f"{_fmt_time(fe)}<00:00"
        else:
            fe    = time.time() - fs["t_start"]
            fr    = proc / max(fe, 1e-9)
            feta  = (total - proc) / max(fr, 1e-9)
            ftime = f"{_fmt_time(fe)}<{_fmt_time(feta)}"

        tots = f"{total:,}" if total > 0 else "?"

        line = Text(no_wrap=True)
        line.append(
            f"{fname:<31}",
            style="bold bright_white" if active else ("bright_green" if done else "white"),
        )
        line.append(f" | DEI: {fs['dei']:<5}",         style="bright_green")
        line.append(f" | Non-DEI: {fs['non_dei']:<6}",  style="white")
        line.append(
            f" | Fail: {fs['fail']:<2}",
            style="red" if fs["fail"] else "dim",
        )
        line.append(f" | {pct:3.0f}%  ", style="bold" if active else "dim")
        line.append_text(_bar(pct, width=20, done=done, active=active))
        line.append(f"  {proc:,}/{tots} [{ftime}]", style="dim")
        root.add_row(line)

    root.add_row(sep)
    return root


# ── Session state ──────────────────────────────────────────────────────────────
session_start      = time.time()
session_total      = 0
session_dei        = 0
session_non_dei    = 0
session_fail       = 0
_oom_downgraded    = False
_stop_requested    = False
visible_count      = 0
active_stat_idx    = 0

# Track the active batch size globally so it remembers the sweet spot across files
current_batch_size = STARTING_BATCH_SIZE

# Rolling speed window
_speed_t0    = time.time()
_speed_count = 0
_tweets_per_sec = 0.0

overall: dict = {
    "total":             overall_total,
    "processed":         overall_processed,
    "dei":               sum(t["cur_dei_1"] for t in file_tasks),
    "non_dei":           sum(t["cur_dei_0"] for t in file_tasks),
    "fail":              0,
    "completed_files":   0,
    "session_processed": 0,
}

def _refresh(live):
    live.update(_build_display(
        file_stats, overall, active_stat_idx, session_start,
        visible_count, current_batch_size, _oom_downgraded,
        _tweets_per_sec, prior_done_count, len(file_tasks),
    ))


# ── Inference loop ─────────────────────────────────────────────────────────────
_bypass_ok = globals().get("_LM_HEAD_BYPASS_OK", _lm_head_w2 is not None)

try:
    with Live(
        _build_display(
            file_stats, overall, active_stat_idx, session_start,
            visible_count, current_batch_size, _oom_downgraded,
            _tweets_per_sec, prior_done_count, len(file_tasks),
        ),
        console=console,
        refresh_per_second=4,
        transient=False,
    ) as live:

        for task_i, task in enumerate(file_tasks):
            if _stop_requested: break
            if task["prior_done"]: continue

            stat_i = task_to_stat_idx[task_i]
            visible_count  += 1
            active_stat_idx = stat_i
            file_stats[stat_i]["t_start"] = time.time()
            _refresh(live)

            df_to_process = task["df"].iloc[task["start_idx"]:]
            file_exists   = task["start_idx"] > 0

            i = 0
            while i < len(df_to_process):
                if _stop_requested: break

                # Slicing uses the globally tuned current_batch_size
                batch_df = df_to_process.iloc[i : i + current_batch_size].copy()
                prompts  = [
                    format_inference_prompt(clean_tweet(tw))
                    for tw in batch_df["Tweet"].tolist()
                ]

                try:
                    enc = tokenizer(
                        prompts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=768,
                        pad_to_multiple_of=8,
                    ).to(inf_model.device)

                    with torch.inference_mode():
                        if _bypass_ok and _lm_head_w2 is not None:
                            transformer_out = inf_model.base_model.model.model(**enc)
                            last_hidden     = transformer_out.last_hidden_state[:, -1, :]
                            del transformer_out
                            logits_01 = (last_hidden @ _lm_head_w2.T).float()
                            del last_hidden, enc
                        else:
                            outputs   = inf_model(**enc)
                            logits_01 = outputs.logits[:, -1, [T0_ID, T1_ID]].float()
                            del outputs, enc

                        probs = torch.softmax(logits_01, dim=-1)
                        preds = (probs[:, 1] >= INFERENCE_THRESHOLD).int().tolist()
                        del logits_01, probs

                    # Write results
                    batch_df["dei_label"] = preds
                    batch_df = batch_df.drop(columns=["char_len", "orig_idx"], errors="ignore")
                    batch_df.to_csv(
                        task["output_path"],
                        mode="a",
                        header=not file_exists,
                        index=False,
                        encoding="utf-8",
                    )
                    file_exists = True

                    b_dei = sum(preds)
                    b_non = len(preds) - b_dei
                    n     = len(preds)

                    file_stats[stat_i]["dei"]       += b_dei
                    file_stats[stat_i]["non_dei"]   += b_non
                    file_stats[stat_i]["processed"] += n
                    overall["dei"]                  += b_dei
                    overall["non_dei"]              += b_non
                    overall["processed"]            += n
                    overall["session_processed"]    += n
                    session_total                   += n
                    session_dei                     += b_dei
                    session_non_dei                 += b_non

                    # Rolling speed window — refresh every 15 s for tighter feedback
                    _speed_count += n
                    _speed_elapsed = time.time() - _speed_t0
                    if _speed_elapsed >= 15.0:
                        _tweets_per_sec = _speed_count / _speed_elapsed
                        _speed_t0       = time.time()
                        _speed_count    = 0

                    _batch_num = file_stats[stat_i]["processed"] // max(current_batch_size, 1)
                    if _batch_num % 10 == 0:
                        torch.cuda.empty_cache()

                    i += n

                except torch.cuda.OutOfMemoryError:
                    # Clean up partial allocations gracefully
                    for _var in ("enc", "transformer_out", "last_hidden",
                                 "outputs", "logits_01", "probs"):
                        try: del locals()[_var]
                        except KeyError: pass
                    gc.collect()
                    torch.cuda.empty_cache()

                    if current_batch_size == 1:
                        # Cannot drop lower than 1. Mark as failed and skip.
                        file_stats[stat_i]["fail"] += 1
                        overall["fail"]            += 1
                        session_fail               += 1
                        i += 1
                    else:
                        # SMART DIAL-DOWN: Scale back by ~20% instead of dropping completely to 16.
                        # This finds the EXACT threshold your 14.6 GB card can handle.
                        current_batch_size = max(1, int(current_batch_size * 0.8))
                        _oom_downgraded = True
                        # Notice we do NOT increment `i` here. The loop will restart
                        # and try to process the EXACT same tweets with the newly reduced batch size.

                _refresh(live)

            # Freeze file timer
            file_stats[stat_i]["done"]  = (
                file_stats[stat_i]["processed"] >= file_stats[stat_i]["total"]
            )
            file_stats[stat_i]["t_end"] = time.time()
            overall["completed_files"] += 1
            _refresh(live)

        _refresh(live)

except KeyboardInterrupt:
    _stop_requested = True
    console.print(
        "\n🛑 Interrupted — progress saved. Re-run Block 11d to resume.",
        style="bold red",
    )

# ── Session Summary ────────────────────────────────────────────────────────────
_elapsed     = int(time.time() - session_start)
_mins, _secs = divmod(_elapsed, 60)
_hrs,  _mins = divmod(_mins, 60)
time_str     = f"{_hrs}h {_mins}m {_secs}s" if _hrs else f"{_mins}m {_secs}s"

tps_final = session_total / max(_elapsed, 1)
tpm_final = tps_final * 60
tph_final = tps_final * 3600

grand_total = max(0, min(overall["processed"], overall_total))
grand_dei   = overall["dei"]
grand_non   = overall["non_dei"]
grand_remaining = max(0, overall_total - grand_total)

console.print()
console.print("=" * 70, style="grey30")
console.print("🚀 SESSION SUMMARY", style="bold green")
console.print("=" * 70, style="grey30")
console.print(f"⏱️  Time              : {time_str}")
console.print(f"⚡ Speed             : {tps_final:.1f} tweets/sec  |  {tpm_final:,.0f} tweets/min  |  {tph_final:,.0f} tweets/hr")
console.print(f"✅ This session      : {session_total:,} tweets")
console.print(f"   • DEI            : {session_dei:,}",      style="bright_green")
console.print(f"   • Non-DEI        : {session_non_dei:,}",  style="white")
if session_fail:
    console.print(f"   • Failed/Skip    : {session_fail}",   style="red")
if _oom_downgraded:
    console.print(
        f"⚠️  Batch size auto-tuned to {current_batch_size} to maximize VRAM.", style="yellow"
    )
console.print("─" * 70, style="grey30")
console.print(f"📊 Grand total       : {grand_total:,} / {overall_total:,} tweets")
if grand_total > 0:
    console.print(
        f"   • DEI            : {grand_dei:,}  ({grand_dei / grand_total * 100:.1f}%)",
        style="bright_green",
    )
    console.print(
        f"   • Non-DEI        : {grand_non:,}  ({grand_non / grand_total * 100:.1f}%)",
        style="white",
    )
console.print(f"   • Remaining       : {grand_remaining:,}")
console.print("=" * 70, style="grey30")

if grand_remaining == 0 and overall_total > 0:
    console.print("🏁 All tweets classified!", style="bold green")
elif session_total > 0:
    console.print("💾 Progress saved — re-run Block 11d to resume.", style="bold")

Output()

# 11d

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK C — Plotting (Final Clean Edition)
# - Sleek callout boxes for top 3 peaks (no floating red text)
# - Restored original, clean legend format
# - Added real-time progress tracking & ETA calculation
# - Background gray plot for Total Tweets included
# ════════════════════════════════════════════════════════════════════════════

import os
import time
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from google.colab import drive
import collections

drive.mount("/content/drive")

BASE_DIR             = "/content/drive/MyDrive/Twitter/Finetune"
INFERENCE_OUTPUT_DIR = os.path.join(BASE_DIR, "inference", "output")
PLOTS_BASE_DIR       = os.path.join(BASE_DIR, "inference", "plots")

MODELS_CONFIG = {
    "Qwen 3.5 4B": {
        "pred_col": "dei_label",
        "color":    "#3274A1", # Professional blue
        "marker":   "o",       # Restored markers
    },
    "Gemini 3 Flash": {
        "output_base": "/content/drive/MyDrive/Twitter/New_everything/Colab/Files/gemini-3-flash-preview/merge",
        "pred_col":    "gemini_label",
        "color":       "#3A923A", # Professional green
        "marker":      "^",
    },
}

CATEGORY_DIR_MAP = {
    "Main": "Main", "Engineering": "Engineering", "Law": "Law",
    "Medical": "Medical", "Business": "Business",
}
CATEGORIES = list(CATEGORY_DIR_MAP.keys())

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def _qwen_cat_dir(category: str) -> str | None:
    for p in [os.path.join(INFERENCE_OUTPUT_DIR, "qwen3.5_4b", CATEGORY_DIR_MAP[category]),
              os.path.join(INFERENCE_OUTPUT_DIR, CATEGORY_DIR_MAP[category])]:
        if os.path.isdir(p): return p
    return None

def _load_classified(path: str, pred_col: str):
    if not os.path.isfile(path): return None
    try: df = pd.read_csv(path, encoding_errors="replace")
    except: return None
    if pred_col not in df.columns or "Date" not in df.columns: return None

    df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce").fillna(0).astype(int)
    df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
    return df.dropna(subset=["Year"]).astype({"Year": int})

def _get_title(df, filename: str) -> str:
    clean = filename.replace("classified_", "").replace(".csv", "")
    parts = clean.split("_")
    rank = parts[0] if parts[0].isdigit() else ""
    uni = str(df["University"].dropna().unique()[0]) if df is not None and "University" in df.columns and len(df["University"].dropna()) > 0 else (parts[1] if len(parts) > 1 else "")
    return f"{rank} {uni}".strip()

# ─────────────────────────────────────────────────────────────────────────────
# Plot Saving Function
# ─────────────────────────────────────────────────────────────────────────────
def _save_plot(title: str, series: dict, save_dir: str, filename: str) -> None:
    all_years = set()
    for df in series.values():
        if df is not None and not df.empty: all_years.update(df["Year"].unique())
    if not all_years: return

    sorted_years = sorted(all_years)
    fig, ax = plt.subplots(figsize=(14, 7))

    has_total = any("TotalCount" in df.columns for df in series.values() if df is not None)
    ax2 = ax.twinx() if has_total else None
    plotted = False

    for model_name, df in series.items():
        if df is None or df.empty: continue

        cfg = MODELS_CONFIG[model_name]
        df_r = df.set_index("Year").reindex(sorted_years, fill_value=0).reset_index()

        dei_count = int(df_r["Count"].sum())
        total = int(df_r["TotalCount"].sum()) if "TotalCount" in df_r.columns else 0
        pct = (dei_count / total * 100) if total > 0 else 0.0

        # RESTORED ORIGINAL LEGEND FORMAT
        label = f"{model_name}  ({dei_count:,} / {total:,}  {pct:.1f}% DEI)"

        # Plot Line with Standard Markers
        ax.plot(
            df_r["Year"], df_r["Count"],
            marker=cfg["marker"], color=cfg["color"],
            linewidth=2, markersize=8, label=label, zorder=3
        )

        # Sleek Annotations for Top 3 Peaks
        if model_name == "Qwen 3.5 4B":
            top3 = df_r.nlargest(3, "Count")
            for _, row in top3.iterrows():
                if row['Count'] > 0:
                    bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="darkgray", lw=1, alpha=0.9)
                    ax.annotate(
                        f"{int(row['Count'])}",
                        xy=(row['Year'], row['Count']),
                        xytext=(0, 15), textcoords='offset points',
                        ha='center', va='bottom',
                        color='black', fontsize=11, fontweight='500',
                        bbox=bbox_props, zorder=5
                    )

        # Background Total tweets
        if ax2 and "TotalCount" in df_r.columns and model_name == "Qwen 3.5 4B":
            ax2.fill_between(df_r["Year"], 0, df_r["TotalCount"], color="#e0e0e0", alpha=0.4, zorder=1)
            ax2.plot(df_r["Year"], df_r["TotalCount"], color="#a0a0a0", linestyle="--", linewidth=1.5, label="Total Tweets (All Topics)")

        plotted = True

    if not plotted:
        plt.close()
        return

    ax.set_title(title, fontsize=16, fontweight="bold", pad=20, color="#333333")
    ax.set_xlabel("Year", fontsize=12, fontweight="500", color="#555555")
    ax.set_ylabel("DEI Tweet Volume (Count)", fontsize=12, fontweight="500", color="#555555")
    ax.set_xticks(sorted_years)
    ax.tick_params(axis="x", rotation=45, colors="#333333")
    ax.tick_params(axis="y", colors="#333333")
    ax.grid(True, linestyle=":", alpha=0.6, zorder=0)

    ax.spines['top'].set_visible(False)
    if ax2: ax2.spines['top'].set_visible(False)

    if ax2:
        ax2.set_ylabel("Total Overall Tweets (Background)", fontsize=12, fontweight="500", color="#888888")
        ax2.tick_params(axis='y', labelcolor="#888888")
        ax2.set_ylim(bottom=0)
        lines_1, labels_1 = ax.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", fontsize=11, framealpha=0.9, edgecolor="#dddddd")
    else:
        ax.legend(loc="upper left", fontsize=11)

    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, f"{filename.replace('.csv','').replace('/','_').replace(' ','_')}.png"), dpi=200, bbox_inches="tight")
    plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# Accumulators & Main Loop (WITH PROGRESS TRACKING)
# ─────────────────────────────────────────────────────────────────────────────
global_acc = {m: {cat: [] for cat in CATEGORIES} for m in MODELS_CONFIG}
uni_acc    = {m: {} for m in MODELS_CONFIG}
uni_cats   = collections.defaultdict(set)

print(f"{'='*60}\nStarting Processing...\n{'='*60}")

for category in CATEGORIES:
    cat_dir = _qwen_cat_dir(category)
    if not cat_dir: continue

    plot_cat_dir = os.path.join(PLOTS_BASE_DIR, "Qwen3.5_4b", category)
    os.makedirs(plot_cat_dir, exist_ok=True)

    qwen_files = sorted(f for f in os.listdir(cat_dir) if f.lower().endswith(".csv"))
    total_files = len(qwen_files)

    print(f"\n📁 [{category}] Total files to process: {total_files}")

    start_time = time.time()

    for idx, fname in enumerate(qwen_files, 1):
        # ─── Load Qwen Data ───
        qwen_cfg = MODELS_CONFIG["Qwen 3.5 4B"]
        df_qwen = _load_classified(os.path.join(cat_dir, fname), qwen_cfg["pred_col"])
        if df_qwen is None: continue

        base_title = _get_title(df_qwen, fname)
        uni_cats[base_title].add(category)

        # ─── Aggregate Data ───
        agg_qwen = df_qwen.groupby("Year").agg(
            Count=(qwen_cfg["pred_col"], "sum"),
            TotalCount=("Year", "size")
        ).reset_index()

        global_acc["Qwen 3.5 4B"][category].append(agg_qwen)

        if base_title not in uni_acc["Qwen 3.5 4B"]:
            for m in MODELS_CONFIG: uni_acc[m][base_title] = []

        uni_acc["Qwen 3.5 4B"][base_title].append(agg_qwen)
        series = {"Qwen 3.5 4B": agg_qwen}

        # ─── Load Gemini Data (Optional) ───
        gem_cfg = MODELS_CONFIG["Gemini 3 Flash"]
        df_gem = _load_classified(os.path.join(gem_cfg["output_base"], fname.replace("classified_", "", 1)), gem_cfg["pred_col"])

        if df_gem is not None:
            agg_gem = df_gem.groupby("Year").agg(
                Count=(gem_cfg["pred_col"], "sum"),
                TotalCount=("Year", "size")
            ).reset_index()
            series["Gemini 3 Flash"] = agg_gem
            global_acc["Gemini 3 Flash"][category].append(agg_gem)
            uni_acc["Gemini 3 Flash"][base_title].append(agg_gem)
        else:
            series["Gemini 3 Flash"] = pd.DataFrame()

        # ─── Save individual plot ───
        _save_plot(f"{base_title}  [{category}]", series, plot_cat_dir, fname.replace("classified_", "", 1))

        # ─── Calculate ETA and print progress ───
        elapsed = time.time() - start_time
        avg_time = elapsed / idx
        remaining_files = total_files - idx
        eta_seconds = remaining_files * avg_time

        # Format times nicely (Mins:Secs)
        el_m, el_s = divmod(int(elapsed), 60)
        eta_m, eta_s = divmod(int(eta_seconds), 60)

        # Overwrite the current line with \r so it doesn't spam the console
        print(f"\r  └─ Progress: [{idx}/{total_files}] | Elapsed: {el_m:02d}:{el_s:02d} | ETA: {eta_m:02d}:{eta_s:02d} | Current: {base_title[:25]:<25}", end="", flush=True)

    print() # Newline after the category loop finishes

    # Category Aggregation Plot
    agg_dir = os.path.join(PLOTS_BASE_DIR, "Qwen3.5_4b", "Aggregated")
    cat_series = {}
    for model_name in MODELS_CONFIG:
        dfs = global_acc[model_name][category]
        cat_series[model_name] = pd.concat(dfs).groupby("Year").sum().reset_index() if dfs else pd.DataFrame()
    _save_plot(f"All Universities — [{category} Only]", cat_series, agg_dir, f"Aggregated_{category}")


# ─────────────────────────────────────────────────────────────────────────────
# Final Aggregations (Universities Cross-Category & Global Overall)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[Aggregating Combined University Data & Generating Summaries]")
uni_agg_dir = os.path.join(PLOTS_BASE_DIR, "Qwen3.5_4b", "Aggregated_University")
os.makedirs(uni_agg_dir, exist_ok=True)

uni_count = len(uni_acc["Qwen 3.5 4B"])
for i, (uni_title, models_dfs) in enumerate(uni_acc["Qwen 3.5 4B"].items(), 1):
    u_series = {m: pd.concat(uni_acc[m][uni_title]).groupby("Year").sum().reset_index() if uni_acc[m].get(uni_title) else pd.DataFrame() for m in MODELS_CONFIG}

    if not u_series["Qwen 3.5 4B"].empty:
        cats_found = ", ".join(sorted(list(uni_cats[uni_title])))
        _save_plot(f"{uni_title}  [{cats_found}]", u_series, uni_agg_dir, f"Aggregated_{uni_title}")

    print(f"\r  └─ Merging Universities: [{i}/{uni_count}]", end="", flush=True)

print()

# Overall Global Aggregation Plot
overall_series = {m: pd.concat([df for cat in CATEGORIES for df in global_acc[m][cat]]).groupby("Year").sum().reset_index() if any(global_acc[m].values()) else pd.DataFrame() for m in MODELS_CONFIG}
_save_plot("Total DEI Tweet Count — All Universities & Categories Combined", overall_series, os.path.join(PLOTS_BASE_DIR, "Qwen3.5_4b", "Aggregated"), "Aggregated_ALL")

print(f"\n{'='*75}")
print(f"✅ All processing complete! Plots generated successfully.")
print(f"{'='*75}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting Processing...

📁 [Main] Total files to process: 20
  └─ Progress: [20/20] | Elapsed: 00:24 | ETA: 00:00 | Current: 9 California Institute of

📁 [Engineering] Total files to process: 32
  └─ Progress: [32/32] | Elapsed: 00:27 | ETA: 00:00 | Current: 9 California Institute of

📁 [Law] Total files to process: 22
  └─ Progress: [22/22] | Elapsed: 00:19 | ETA: 00:00 | Current: 8 University of Pennsylva

📁 [Medical] Total files to process: 25
  └─ Progress: [25/25] | Elapsed: 00:27 | ETA: 00:00 | Current: 8 University of Pennsylva

📁 [Business] Total files to process: 10
  └─ Progress: [10/10] | Elapsed: 00:10 | ETA: 00:00 | Current: 8 University of Pennsylva

[Aggregating Combined University Data & Generating Summaries]
  └─ Merging Universities: [35/35]

✅ All processing complete! Plots generated successfully.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')


def generate_tree_and_size(startpath):
    total_size_mb = 0

    print(f"📂 {os.path.basename(startpath)}/")

    for root, dirs, files in os.walk(startpath):
        # Exclude hidden Colab/Git folders
        dirs[:] =[d for d in dirs if not d.startswith('.')]

        level = root.replace(startpath, '').count(os.sep)
        indent = '│   ' * level + '├── '

        # Don't print the root folder again
        if root != startpath:
            print(f"{'│   ' * (level - 1)}├── 📂 {os.path.basename(root)}/")

        subindent = '│   ' * (level + 1) + '├── '

        for f in files:
            fp = os.path.join(root, f)
            if not os.path.islink(fp):
                size_mb = os.path.getsize(fp) / (1024 * 1024)
                total_size_mb += size_mb

                # To prevent spamming the console, we won't print every single CSV
                # if there are hundreds, but we will print model files and checkpoints
                if f.endswith('.safetensors') or f.endswith('.json') or f.endswith('.pt') or f.endswith('.csv') or f.endswith('.png'):
                    print(f"{subindent}📄 {f} ({size_mb:.2f} MB)")
                else:
                    print(f"{subindent}📄 {f} ({size_mb:.2f} MB)")

    print("\n" + "="*50)
    print(f"Total Google Drive Space Used: {total_size_mb:.2f} MB")
    print("="*50)

# Run the function
drive_path = '/content/drive/MyDrive/Twitter/Finetune'
if os.path.exists(drive_path):
    generate_tree_and_size(drive_path)
else:
    print(f"Path not found: {drive_path}. Please ensure Drive is mounted.")

In [ ]:
import os
import pandas as pd

# ==========================================
# 1. SETUP CONFIGURATION
# ==========================================
# Directory fix: saving directly to the Qwen3.5_9b subfolder
output_dir = '/content/drive/MyDrive/Twitter/Finetune/inference/plots/Qwen3.5_9b'
os.makedirs(output_dir, exist_ok=True)
output_excel_path = os.path.join(output_dir, 'Model_Disagreement_Analysis.xlsx')

MODELS_CONFIG = {
    "Qwen_4B": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_4b",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label"
    },
    "Qwen_9B": {
        "folder": "/content/drive/MyDrive/Twitter/Finetune/inference/output/qwen3.5_9b",
        "file_prefix": "classified_",
        "file_suffix": ".csv",
        "pred_col": "dei_label"
    },
    "Gemini": {
        "folder": "/content/drive/MyDrive/Twitter/New_everything/Colab/Files/gemini-3-flash-preview/merge",
        "file_prefix": "",
        "file_suffix": ".csv",
        "pred_col": "gemini_label"
    }
}

# ==========================================
# 2. COLLECT DISAGREEMENTS
# ==========================================
disagreements = {
    "Only_Qwen4B_Positive": pd.DataFrame(),
    "Only_Qwen9B_Positive": pd.DataFrame(),
    "Only_Gemini_Positive": pd.DataFrame(),
    "Qwen9B_Neg_Others_Pos": pd.DataFrame() # 9B=0 while 4B=1 AND Gemini=1
}

total_scanned = 0
print(f"Scanning files and aggregating data...\n")

for uni in uni_names:
    p4b = os.path.join(MODELS_CONFIG["Qwen_4B"]["folder"], f"{MODELS_CONFIG['Qwen_4B']['file_prefix']}{uni}.csv")
    p9b = os.path.join(MODELS_CONFIG["Qwen_9B"]["folder"], f"{MODELS_CONFIG['Qwen_9B']['file_prefix']}{uni}.csv")
    pgem = os.path.join(MODELS_CONFIG["Gemini"]["folder"], f"{MODELS_CONFIG['Gemini']['file_prefix']}{uni}.csv")

    if not (os.path.exists(p4b) and os.path.exists(p9b) and os.path.exists(pgem)):
        continue

    df4b, df9b, dfgem = pd.read_csv(p4b), pd.read_csv(p9b), pd.read_csv(pgem)
    total_scanned += len(df4b)

    # --- Find Text Column ---
    text_col = next((c for c in ['Tweet', 'tweet', 'Text', 'text'] if c in df4b.columns), df4b.columns[0])

    combined_df = pd.DataFrame({
        'University': uni,
        'Tweet_Text': df4b[text_col],
        'L_4B': df4b[MODELS_CONFIG['Qwen_4B']['pred_col']],
        'L_9B': df9b[MODELS_CONFIG['Qwen_9B']['pred_col']],
        'L_Gemini': dfgem[MODELS_CONFIG['Gemini']['pred_col']]
    })

    l4, l9, lg = combined_df['L_4B'], combined_df['L_9B'], combined_df['L_Gemini']

    # Logic Slices
    disagreements["Only_Qwen4B_Positive"] = pd.concat([disagreements["Only_Qwen4B_Positive"], combined_df[(l4==1) & (l9==0) & (lg==0)]])
    disagreements["Only_Qwen9B_Positive"] = pd.concat([disagreements["Only_Qwen9B_Positive"], combined_df[(l9==1) & (l4==0) & (lg==0)]])
    disagreements["Only_Gemini_Positive"] = pd.concat([disagreements["Only_Gemini_Positive"], combined_df[(lg==1) & (l4==0) & (l9==0)]])
    disagreements["Qwen9B_Neg_Others_Pos"] = pd.concat([disagreements["Qwen9B_Neg_Others_Pos"], combined_df[(l9==0) & (l4==1) & (lg==1)]])

# ==========================================
# 3. STATS OUTPUT & EXPORT
# ==========================================
print("--- ANALYSIS SUMMARY ---")
print(f"Total Processed: {total_scanned:,} tweets\n")

summary_data = []
for key, df in disagreements.items():
    count = len(df)
    perc = (count / total_scanned * 100) if total_scanned > 0 else 0
    print(f"{key: <25} : {count: >8,} ({perc:.2f}%)")
    summary_data.append({"Category": key, "Count": count, "Percentage": f"{perc:.2f}%"})

with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    # Add a summary sheet first
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)
    # Add sampled disagreement sheets
    for sheet, df in disagreements.items():
        if not df.empty:
            df.sample(n=min(100, len(df)), random_state=42).to_excel(writer, sheet_name=sheet[:31], index=False)

print(f"\nSuccess! Results saved to:\n{output_excel_path}")